In [1]:
# ============================================================
import pandas as pd
import numpy as np
import logging
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# 导入自定义模块
from src.core.database import DatabaseManager
from src.core.data_fetcher import DataFetcher
from src.core.spread_calculator import SpreadCalculator
from src.core.indicators import IndicatorBuilder
from src.core.feature_engineering import FeatureEngineer
from src.core.ml_models import MLModel, SignalGenerator
from src.core.backtest import BacktestEngine, PerformanceAnalyzer
from src.core.visualization import Visualizer

# ============================================================
# 配置日志
# ============================================================
log_dir = Path('logs')
log_dir.mkdir(exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(log_dir / 'debug_strategy.log', encoding='utf-8'),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

logger.info("="*60)
logger.info("开始运行调试版本策略")
logger.info("="*60)

# ============================================================
# 全局变量初始化
# ============================================================
print("\n[1/9] 初始化模块...")

# 数据库和工具类
db_path = "data/trading_data.db"
db = DatabaseManager(db_path)
fetcher = DataFetcher()
spread_calc = SpreadCalculator(db)
indicator_builder = IndicatorBuilder(db)
feature_engineer = FeatureEngineer(indicator_builder)
visualizer = Visualizer(output_dir="outputs/Palm&Soybean_Results")

# 数据存储容器
price_data = {}          # 存储各品种价格数据
spread_data = {}         # 存储价差数据
macro_data = {}          # 存储宏观数据
fundamental_data = {}    # 存储基本面数据

# 特征和模型
features_df = None       # 合并后的特征DataFrame
X_train = None          # 训练集特征
X_test = None           # 测试集特征
y_train = None          # 训练集标签
y_test = None           # 测试集标签
train_idx = None        # 训练集索引
test_idx = None         # 测试集索引
model = None            # 训练好的模型
selected_features = []  # 选择的特征列表

# 回测结果
signals = None          # 交易信号
equity_curve = None     # 权益曲线
trade_log = None        # 交易日志
performance_report = None  # 绩效报告

print("✓ 模块初始化完成")

START_DATE = "2000-01-01"

INFO:__main__:============================================================
INFO:__main__:开始运行调试版本策略
INFO:__main__:============================================================
INFO:src.core.database:数据表创建完成
INFO:src.core.database:数据库已初始化: data/trading_data.db (timeout=30.0s)



[1/9] 初始化模块...
✓ 模块初始化完成


In [2]:
# ============================================================
# 步骤1.1：获取中国豆油棕榈油期货数据
# ============================================================
print("\n[2.1/9] 开始获取中国豆油棕榈油及上下游商品期货数据...")
logger.info("\n" + "="*60)
logger.info("步骤1.1：获取中国期货数据")
logger.info("="*60)

# 中国期货品种配置
# 豆油主力合约代码：JM0（大连商品交易所）
# 焦炭主力合约代码：J0（大连商品交易所）
cn_symbols_config = {
    'P': 'P0',    # 棕榈油主力合约
    'Y': 'Y0',       #豆油主力合约
    'B': 'B0',     # 豆一主力合约
    'A':'A0'# 豆二主力合约
}

# 获取中国期货数据
print("  正在从akshare获取中国期货数据...")
for name, symbol in cn_symbols_config.items():
    print(f"  获取 {name} ({symbol}) 数据...")
    logger.info(f"获取 {name} 数据...")
    

    df = fetcher.fetch_akshare_futures_data(symbol, market='CN')

    # 重命名列为英文（akshare返回的是中文列名）

    df.index = pd.to_datetime(df.index)

    if not df.empty:

        
        df_adjusted = df.copy()
        
        # 保存到数据库
        db.insert_price_data(df_adjusted, name, 'futures')
        
        print(f"  ✓ {name}: {len(df_adjusted)} 条记录")
        logger.info(f"{name} 数据获取成功: {len(df_adjusted)} 条记录")
        
        # 显示数据日期范围
        if hasattr(df_adjusted.index, 'min') and hasattr(df_adjusted.index, 'max'):
            date_range = f"{df_adjusted.index.min()} 至 {df_adjusted.index.max()}"
            print(f"      日期范围: {date_range}")
            print(f"      时区: {df_adjusted.index.tz}")
    else:
        print(f"  ✗ {name} 数据获取失败")
        logger.warning(f"{name} 数据获取失败")
            


print("\n✓ 中国期货数据获取完成")
logger.info("中国期货数据获取完成\n")

INFO:__main__:
INFO:__main__:步骤1.1：获取中国期货数据
INFO:__main__:============================================================
INFO:__main__:获取 P 数据...



[2.1/9] 开始获取中国豆油棕榈油及上下游商品期货数据...
  正在从akshare获取中国期货数据...
  获取 P (P0) 数据...


INFO:src.core.data_fetcher:正在从akshare获取 P0 的期货数据...
INFO:src.core.data_fetcher:成功获取 4381 条 P0 的数据
INFO:src.core.database:插入了 1 条新数据，跳过了 4380 条重复数据
INFO:__main__:P 数据获取成功: 4381 条记录
INFO:__main__:获取 Y 数据...
INFO:src.core.data_fetcher:正在从akshare获取 Y0 的期货数据...


  ✓ P: 4381 条记录
      日期范围: 2007-10-29 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      时区: Asia/Shanghai
  获取 Y (Y0) 数据...


INFO:src.core.data_fetcher:成功获取 4813 条 Y0 的数据
INFO:src.core.database:插入了 1 条新数据，跳过了 4812 条重复数据
INFO:__main__:Y 数据获取成功: 4813 条记录
INFO:__main__:获取 B 数据...
INFO:src.core.data_fetcher:正在从akshare获取 B0 的期货数据...


  ✓ Y: 4813 条记录
      日期范围: 2006-01-09 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      时区: Asia/Shanghai
  获取 B (B0) 数据...


INFO:src.core.data_fetcher:成功获取 4272 条 B0 的数据
INFO:src.core.database:插入了 1 条新数据，跳过了 4271 条重复数据
INFO:__main__:B 数据获取成功: 4272 条记录
INFO:__main__:获取 A 数据...
INFO:src.core.data_fetcher:正在从akshare获取 A0 的期货数据...


  ✓ B: 4272 条记录
      日期范围: 2007-08-27 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      时区: Asia/Shanghai
  获取 A (A0) 数据...


INFO:src.core.data_fetcher:成功获取 5067 条 A0 的数据
INFO:src.core.database:插入了 1 条新数据，跳过了 5066 条重复数据
INFO:__main__:A 数据获取成功: 5067 条记录
INFO:__main__:中国期货数据获取完成



  ✓ A: 5067 条记录
      日期范围: 2005-01-04 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      时区: Asia/Shanghai

✓ 中国期货数据获取完成


In [3]:
# ============================================================
# 步骤1.2：从数据库提取豆油和焦炭数据（不使用内存数据）
# ============================================================
print("\n[2.2/9] 开始从数据库提取豆油棕榈油数据...")
logger.info("\n" + "="*60)
logger.info("步骤1.2：从数据库提取中国期货数据")
logger.info("="*60)

# ✅ 清空内存中的数据，强制从数据库读取
print("  清空内存中的临时数据...")
price_data.clear()

# 1. 提取豆油和棕榈油期货价格数据
print("  [1.2.1] 从数据库提取豆油棕榈油期货价格数据...")
cn_symbols = ['P', 'Y']

for symbol in cn_symbols:
    try:
        df = db.get_price_data(
            symbol=symbol,
            start_date=START_DATE,
            columns=['date', 'open', 'high', 'low', 'close', 'volume']
        )
        
        if not df.empty:
            # 设置索引
            if 'date' in df.columns:
                df.set_index('date', inplace=True)
            
            price_data[symbol] = df
            date_range = f"{df.index.min()} 至 {df.index.max()}"
            print(f"    ✓ {symbol}: {len(df)} 条记录, 日期范围: {date_range}")
            logger.info(f"{symbol} 数据提取成功: {len(df)} 条记录")
        else:
            print(f"    ✗ {symbol}: 数据库中无数据")
            logger.warning(f"{symbol} 数据库中无数据，请先运行数据获取步骤")
            
    except Exception as e:
        print(f"    ✗ {symbol}: 提取失败 - {e}")
        logger.error(f"{symbol} 提取失败: {e}")

# 2. 数据完整性检查
print("\n  [1.2.2] 数据完整性检查...")
required_symbols = ['P', 'Y']
missing_symbols = [s for s in required_symbols if s not in price_data or price_data[s].empty]

if missing_symbols:
    error_msg = f"缺少必要的期货数据: {missing_symbols}"
    print(f"    ❌ {error_msg}")
    logger.error(error_msg)
    print("\n    💡 解决方案:")
    print("    1. 运行上一个代码单元格获取数据")
    print("    2. 检查网络连接和akshare库是否正常")
    print("    3. 查看日志文件: logs/debug_strategy.log")
else:
    print("    ✅ 焦煤和焦炭数据已就绪")
    
    # 显示数据统计信息
    print("\n  [1.2.3] 数据统计信息:")
    for symbol in required_symbols:
        if symbol in price_data:
            df = price_data[symbol]
            print(f"    {symbol}:")
            print(f"      数据量: {len(df)} 条")
            print(f"      日期范围: {df.index.min()} 至 {df.index.max()}")
            print(f"      时区信息: {df.index.tz}")
            print(f"      价格范围: {df['close'].min():.2f} - {df['close'].max():.2f}")
            print(f"      平均成交量: {df['volume'].mean():.0f}")

print("\n✓ 豆油和棕榈油数据提取完成（数据来源：数据库）")
logger.info("豆油和棕榈油数据提取完成\n")

INFO:__main__:
INFO:__main__:步骤1.2：从数据库提取中国期货数据
INFO:__main__:============================================================
INFO:__main__:P 数据提取成功: 4381 条记录
INFO:__main__:Y 数据提取成功: 4813 条记录
INFO:__main__:豆油和棕榈油数据提取完成




[2.2/9] 开始从数据库提取豆油棕榈油数据...
  清空内存中的临时数据...
  [1.2.1] 从数据库提取豆油棕榈油期货价格数据...
    ✓ P: 4381 条记录, 日期范围: 2007-10-29 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
    ✓ Y: 4813 条记录, 日期范围: 2006-01-09 00:00:00+08:00 至 2025-11-03 00:00:00+08:00

  [1.2.2] 数据完整性检查...
    ✅ 焦煤和焦炭数据已就绪

  [1.2.3] 数据统计信息:
    P:
      数据量: 4381 条
      日期范围: 2007-10-29 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      时区信息: pytz.FixedOffset(480)
      价格范围: 4020.00 - 12718.00
      平均成交量: 575218
    Y:
      数据量: 4813 条
      日期范围: 2006-01-09 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      时区信息: pytz.FixedOffset(480)
      价格范围: 4996.00 - 14154.00
      平均成交量: 494563

✓ 豆油和棕榈油数据提取完成（数据来源：数据库）


In [4]:
# 获取宏观数据
print("\n  获取宏观经济数据...")
logger.info("获取宏观经济数据...")

# VIX波动率指数
vix_data = fetcher.fetch_index_data('VIX', start_date=START_DATE)
if not vix_data.empty:
    db.insert_price_data(vix_data, 'VIX', 'index')
    macro_data['VIX'] = vix_data
    print(f"  ✓ VIX: {len(vix_data)} 条记录")
    logger.info(f"VIX数据获取成功: {len(vix_data)} 条记录")

# 美元指数
dxy_data = fetcher.fetch_index_data('DXY', start_date=START_DATE)
if not dxy_data.empty:
    db.insert_price_data(dxy_data, 'DXY', 'index')
    macro_data['DXY'] = dxy_data
    print(f"  ✓ DXY: {len(dxy_data)} 条记录")
    logger.info(f"DXY数据获取成功: {len(dxy_data)} 条记录")

INFO:__main__:获取宏观经济数据...



  获取宏观经济数据...


INFO:src.core.data_fetcher:正在从yfinance获取 ^VIX 的数据...
INFO:src.core.data_fetcher:成功获取 6498 条 ^VIX 的数据
INFO:src.core.database:插入了 0 条新数据，跳过了 6498 条重复数据
INFO:__main__:VIX数据获取成功: 6498 条记录
INFO:src.core.data_fetcher:正在从yfinance获取 DX-Y.NYB 的数据...


  ✓ VIX: 6498 条记录


INFO:src.core.data_fetcher:成功获取 6527 条 DX-Y.NYB 的数据
INFO:src.core.database:插入了 0 条新数据，跳过了 6527 条重复数据
INFO:__main__:DXY数据获取成功: 6527 条记录


  ✓ DXY: 6527 条记录


## 📊 步骤1.2.6：从数据库提取大豆价格数据

从数据库中提取大豆数据，作为行业上游指标使用

In [5]:
# ============================================================
# 步骤1.2.6：从数据库提取大豆价格数据（按宏观数据方式处理）
# ============================================================
print("\n[2.2.6/9] 从数据库提取大豆数据（上游行业指标）...")
logger.info("\n" + "="*60)
logger.info("步骤1.2.6：提取大豆数据")
logger.info("="*60)

# 提取大豆数据（作为宏观/行业指标处理）
print("  [1.2.6.1] 提取大豆价格数据...")
upstream_symbols = ['A', 'B']

for symbol in upstream_symbols:
    try:
        df = db.get_price_data(
            symbol=symbol,
            start_date=START_DATE,
            columns=['date', 'close', 'volume', 'open', 'high', 'low']
        )
        
        if not df.empty:
            # 设置索引
            if 'date' in df.columns:
                df.set_index('date', inplace=True)
            
            # 确保索引有时区（与宏观数据保持一致）
            if hasattr(df.index, 'tz'):
                if df.index.tz is None:
                    df.index = df.index.tz_localize('UTC')
                    print(f"    ⚠️  {symbol}: 数据库数据无时区，已添加UTC时区")
                else:
                    print(f"    ✓ {symbol}: 时区 = {df.index.tz}")
            
            # 存储到 macro_data 字典（作为行业上游宏观指标）
            macro_data[symbol] = df
            
            date_range = f"{df.index.min()} 至 {df.index.max()}"
            print(f"    ✓ {symbol}: {len(df)} 条记录, 日期范围: {date_range}")
            logger.info(f"{symbol} 数据提取成功: {len(df)} 条记录")
            
            # 显示统计信息
            print(f"      价格范围: {df['close'].min():.2f} - {df['close'].max():.2f}")
            print(f"      平均成交量: {df['volume'].mean():.0f}")
            
        else:
            print(f"    ✗ {symbol}: 数据库中无数据")
            logger.warning(f"{symbol} 数据库中无数据，请先运行数据获取步骤")
            
    except Exception as e:
        print(f"    ✗ {symbol}: 提取失败 - {e}")
        logger.error(f"{symbol} 提取失败: {e}")

# 数据验证
print("\n  [1.2.6.2] 大豆数据验证...")
if 'B' in macro_data and not macro_data['B'].empty:
    b_df = macro_data['B']
    print(f"    ✅ 大豆数据已就绪")
    print(f"    数据量: {len(b_df)} 条")
    print(f"    日期范围: {b_df.index.min()} 至 {b_df.index.max()}")
    print(f"    时区信息: {b_df.index.tz}")

    # 计算与焦炭、焦煤的相关性（如果数据存在）
    if 'P' in price_data and 'Y' in price_data:
        print(f"\n  [1.2.6.3] 相关性分析:")
        # 对齐数据进行相关性分析
        b_close = b_df['close']

        if 'P' in price_data:
            p_close = price_data['P']['close']
            # 找到共同的日期索引
            common_dates = b_close.index.intersection(p_close.index)
            if len(common_dates) > 0:
                corr_jm = b_close.loc[common_dates].corr(p_close.loc[common_dates])
                print(f"      大豆 vs 棕榈油: {corr_jm:.4f}")
        
        if 'Y' in price_data:
            j_close = price_data['Y']['close']
            common_dates = b_close.index.intersection(j_close.index)
            if len(common_dates) > 0:
                corr_j = b_close.loc[common_dates].corr(j_close.loc[common_dates])
                print(f"      大豆 vs 豆油: {corr_j:.4f}")
else:
    print(f"    ⚠️  大豆数据未就绪")
    print(f"    💡 请先运行上一个单元格获取数据")

print("\n✓ 大豆数据提取完成（数据来源：数据库，存储位置：macro_data['B']）")
logger.info("大豆数据提取完成\n")

INFO:__main__:
INFO:__main__:步骤1.2.6：提取大豆数据
INFO:__main__:============================================================
INFO:__main__:A 数据提取成功: 5067 条记录
INFO:__main__:B 数据提取成功: 4272 条记录
INFO:__main__:大豆数据提取完成




[2.2.6/9] 从数据库提取大豆数据（上游行业指标）...
  [1.2.6.1] 提取大豆价格数据...
    ✓ A: 时区 = pytz.FixedOffset(480)
    ✓ A: 5067 条记录, 日期范围: 2005-01-04 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      价格范围: 2421.00 - 6508.00
      平均成交量: 195487
    ✓ B: 时区 = pytz.FixedOffset(480)
    ✓ B: 4272 条记录, 日期范围: 2007-08-27 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      价格范围: 2698.00 - 5748.00
      平均成交量: 36827

  [1.2.6.2] 大豆数据验证...
    ✅ 大豆数据已就绪
    数据量: 4272 条
    日期范围: 2007-08-27 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
    时区信息: pytz.FixedOffset(480)

  [1.2.6.3] 相关性分析:
      大豆 vs 棕榈油: 0.7551
      大豆 vs 豆油: 0.8776

✓ 大豆数据提取完成（数据来源：数据库，存储位置：macro_data['B']）


## 📊 从数据库提取豆棕基本面数据

从数据库提取仓单和库存数据作为基本面指标

## 1.5 从数据库提取豆油和棕榈油基本面数据

In [6]:
# ============================================================
# 步骤1.5：从数据库提取豆油和棕榈油基本面数据
# ============================================================
print("\n[2.5/9] 从数据库提取豆油和棕榈油基本面数据...")
logger.info("\n" + "="*60)
logger.info("步骤1.5：提取豆油和棕榈油基本面数据")
logger.info("="*60)

# 提取大豆和豆油数据
try:
    print("  [1.5.1] 提取 SOYBEAN_INVENTORY 基本面数据...")
    soybean_fundamental = db.get_fundamental_data(
        data_source='SOYBEAN_INVENTORY',
        start_date=START_DATE
    )
    
    if not soybean_fundamental.empty:
        # 解析JSON数据并转换为DataFrame
        import json
        
        print("  [1.5.2] 解析大豆豆油基本面数据...")
        data_list = []
        for _, row in soybean_fundamental.iterrows():
            data_dict = json.loads(row['data_json'])
            data_dict['report_date'] = row['report_date']
            data_list.append(data_dict)
        
        # 创建DataFrame
        soybean_df = pd.DataFrame(data_list)
        soybean_df['report_date'] = pd.to_datetime(soybean_df['report_date'])
        
        # 添加东八区时区
        soybean_df['report_date'] = soybean_df['report_date'].dt.tz_localize('Asia/Shanghai')
        
        # 设置索引
        soybean_df.set_index('report_date', inplace=True)
        
        # 存入fundamental_data字典
        fundamental_data['SOYBEAN_INVENTORY'] = soybean_df
        
        print(f"    ✓ 提取 {len(soybean_df)} 条大豆豆油基本面数据")
        print(f"    ✓ 日期范围: {soybean_df.index.min()} 至 {soybean_df.index.max()}")
        print(f"    ✓ 时区: {soybean_df.index.tz}")
        
        # 显示数据列
        print(f"    ✓ 数据字段:")
        for col in soybean_df.columns:
            non_null = soybean_df[col].notna().sum()
            print(f"      - {col}: {non_null} 个有效值")
        
        # 显示数据样例
        print("\n  [1.5.3] 大豆豆油数据样例（最近10条）:")
        print(soybean_df.tail(10).to_string())
        
    else:
        print("    ⚠️  数据库中无 SOYBEAN_INVENTORY 数据")
        print("    💡 请先运行导入脚本: python outside_data_import/import_soybean_inventory.py")
        logger.warning("SOYBEAN_INVENTORY 数据库中无数据")
        
except Exception as e:
    print(f"    ✗ 大豆豆油数据提取失败: {e}")
    logger.error(f"SOYBEAN_INVENTORY 提取失败: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "-"*60)

# 提取棕榈油数据
try:
    print("  [1.5.4] 提取 PALM_OIL_INVENTORY 基本面数据...")
    palm_oil_fundamental = db.get_fundamental_data(
        data_source='PALM_OIL_INVENTORY',
        start_date=START_DATE
    )
    
    if not palm_oil_fundamental.empty:
        # 解析JSON数据并转换为DataFrame
        import json
        
        print("  [1.5.5] 解析棕榈油基本面数据...")
        data_list = []
        for _, row in palm_oil_fundamental.iterrows():
            data_dict = json.loads(row['data_json'])
            data_dict['report_date'] = row['report_date']
            data_list.append(data_dict)
        
        # 创建DataFrame
        palm_oil_df = pd.DataFrame(data_list)
        palm_oil_df['report_date'] = pd.to_datetime(palm_oil_df['report_date'])
        
        # 添加东八区时区
        palm_oil_df['report_date'] = palm_oil_df['report_date'].dt.tz_localize('Asia/Shanghai')
        
        # 设置索引
        palm_oil_df.set_index('report_date', inplace=True)
        
        # 存入fundamental_data字典
        fundamental_data['PALM_OIL_INVENTORY'] = palm_oil_df
        
        print(f"    ✓ 提取 {len(palm_oil_df)} 条棕榈油基本面数据")
        print(f"    ✓ 日期范围: {palm_oil_df.index.min()} 至 {palm_oil_df.index.max()}")
        print(f"    ✓ 时区: {palm_oil_df.index.tz}")
        
        # 显示数据列
        print(f"    ✓ 数据字段:")
        for col in palm_oil_df.columns:
            non_null = palm_oil_df[col].notna().sum()
            print(f"      - {col}: {non_null} 个有效值")
        
        # 显示数据样例
        print("\n  [1.5.6] 棕榈油数据样例（最近10条）:")
        print(palm_oil_df.tail(10).to_string())
        
    else:
        print("    ⚠️  数据库中无 PALM_OIL_INVENTORY 数据")
        print("    💡 请先运行导入脚本: python outside_data_import/import_palm_oil_inventory.py")
        logger.warning("PALM_OIL_INVENTORY 数据库中无数据")
        
except Exception as e:
    print(f"    ✗ 棕榈油数据提取失败: {e}")
    logger.error(f"PALM_OIL_INVENTORY 提取失败: {e}")
    import traceback
    traceback.print_exc()

print("\n✓ 豆油和棕榈油基本面数据提取完成")
logger.info("豆油和棕榈油基本面数据提取完成\n")

# 显示所有已加载的基本面数据总览
print("\n" + "="*60)
print("📊 所有基本面数据总览:")
print("="*60)
for data_source, df in fundamental_data.items():
    if df is not None and not df.empty:
        print(f"\n✓ {data_source}:")
        print(f"  - 记录数: {len(df)}")
        print(f"  - 日期范围: {df.index.min()} 至 {df.index.max()}")
        print(f"  - 字段: {', '.join(df.columns.tolist())}")
print("="*60)

INFO:__main__:
INFO:__main__:步骤1.5：提取豆油和棕榈油基本面数据
INFO:__main__:============================================================



[2.5/9] 从数据库提取豆油和棕榈油基本面数据...
  [1.5.1] 提取 SOYBEAN_INVENTORY 基本面数据...
  [1.5.2] 解析大豆豆油基本面数据...
    ✓ 提取 3643 条大豆豆油基本面数据
    ✓ 日期范围: 2012-02-24 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
    ✓ 时区: Asia/Shanghai
    ✓ 数据字段:
      - soybean_port_inventory: 3367 个有效值
      - soybean_oil_port_inventory: 1574 个有效值

  [1.5.3] 大豆豆油数据样例（最近10条）:
                           soybean_port_inventory  soybean_oil_port_inventory
report_date                                                                  
2025-10-21 00:00:00+08:00                  729.97                       120.5
2025-10-22 00:00:00+08:00                  751.42                         NaN
2025-10-23 00:00:00+08:00                  773.35                         NaN
2025-10-24 00:00:00+08:00                  811.27                         NaN
2025-10-27 00:00:00+08:00                  814.02                         NaN
2025-10-28 00:00:00+08:00                  824.29                       123.4
2025-10-29 00:00:00+08:00             

INFO:__main__:豆油和棕榈油基本面数据提取完成



  [1.5.5] 解析棕榈油基本面数据...
    ✓ 提取 1896 条棕榈油基本面数据
    ✓ 日期范围: 2013-05-14 00:00:00+08:00 至 2025-10-28 00:00:00+08:00
    ✓ 时区: Asia/Shanghai
    ✓ 数据字段:
      - palm_oil_port_inventory: 1896 个有效值

  [1.5.6] 棕榈油数据样例（最近10条）:
                           palm_oil_port_inventory
report_date                                       
2025-08-26 00:00:00+08:00                     53.4
2025-09-02 00:00:00+08:00                     60.1
2025-09-09 00:00:00+08:00                     64.5
2025-09-16 00:00:00+08:00                     66.5
2025-09-23 00:00:00+08:00                     61.6
2025-09-30 00:00:00+08:00                     58.1
2025-10-14 00:00:00+08:00                     59.8
2025-10-17 00:00:00+08:00                     59.8
2025-10-21 00:00:00+08:00                     62.0
2025-10-28 00:00:00+08:00                     63.9

✓ 豆油和棕榈油基本面数据提取完成

📊 所有基本面数据总览:

✓ SOYBEAN_INVENTORY:
  - 记录数: 3643
  - 日期范围: 2012-02-24 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
  - 字段: soybean_port_inventory, soy

In [7]:
fundamental_data['SOYBEAN_INVENTORY'] = fundamental_data['SOYBEAN_INVENTORY'].ffill().dropna()

In [8]:
print("\n  [1.2] 提取宏观数据...")
macro_symbols = ['VIX', 'DXY']

for symbol in macro_symbols:
    try:
        df = db.get_price_data(
            symbol=symbol,
            start_date=START_DATE,
            columns=['date', 'close']
        )
        
        if not df.empty:
            # 设置索引
            if 'date' in df.columns:
                df.set_index('date', inplace=True)
            
            # 确保索引无时区
            # if hasattr(df.index, 'tz') and df.index.tz is not None:
            #     df.index = df.index.tz_localize(None)
            
            macro_data[symbol] = df
            print(f"    ✓ {symbol}: {len(df)} 条记录")
            logger.info(f"{symbol} 数据提取成功: {len(df)} 条记录")
        else:
            print(f"    ✗ {symbol}: 数据库中无数据")
            logger.warning(f"{symbol} 数据库中无数据")
            
    except Exception as e:
        print(f"    ✗ {symbol}: 提取失败 - {e}")
        logger.error(f"{symbol} 提取失败: {e}")


  [1.2] 提取宏观数据...


INFO:__main__:VIX 数据提取成功: 6498 条记录


    ✓ VIX: 6498 条记录


INFO:__main__:DXY 数据提取成功: 6528 条记录


    ✓ DXY: 6528 条记录


In [9]:

# 4. 数据验证
print("\n  [1.4] 数据验证...")
print("    期货数据:")
for symbol, df in price_data.items():
    date_range = f"{df.index.min()} 至 {df.index.max()}"
    print(f"      {symbol}: {len(df)} 条, 日期范围: {date_range}")

print("    宏观数据:")
for symbol, df in macro_data.items():
    date_range = f"{df.index.min()} 至 {df.index.max()}"
    print(f"      {symbol}: {len(df)} 条, 日期范围: {date_range}")

# if 'EIA' in fundamental_data:
#     eia = fundamental_data['EIA']
#     date_range = f"{eia.index.min()} 至 {eia.index.max()}"
#     print(f"    基本面数据:")
#     print(f"      EIA: {len(eia)} 条, 日期范围: {date_range}")

# 5. 检查数据完整性
print("\n  [1.5] 数据完整性检查...")
required_symbols = ['P', 'Y']
missing_symbols = [s for s in required_symbols if s not in price_data or price_data[s].empty]

if missing_symbols:
    # error_msg = f"缺少必要的期货数据: {missing_symbols}"
    # print(f"    ❌ {error_msg}")
    # logger.error(error_msg)
    # print("\n    💡 解决方案:")
    # print("    1. 运行数据更新脚本: python scripts/daily_data_update.py")
    # print("    2. 检查网络连接和数据源可用性")
    # print("    3. 查看日志文件: logs/daily_update.log")
    raise ValueError(error_msg)
else:
    print("    ✅ 所有必要数据已就绪")

print("\n✓ 数据提取完成")
logger.info("数据提取完成\n")



# 🔍 调试点1：在此处设置断点，检查 price_data, macro_data 的内容
# 可以在调试控制台输入: price_data.keys(), len(price_data['CL'])

INFO:__main__:数据提取完成




  [1.4] 数据验证...
    期货数据:
      P: 4381 条, 日期范围: 2007-10-29 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      Y: 4813 条, 日期范围: 2006-01-09 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
    宏观数据:
      VIX: 6498 条, 日期范围: 2000-01-03 00:00:00-06:00 至 2025-10-31 00:00:00-05:00
      DXY: 6528 条, 日期范围: 2000-01-03 00:00:00-05:00 至 2025-11-02 00:00:00-04:00
      A: 5067 条, 日期范围: 2005-01-04 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
      B: 4272 条, 日期范围: 2007-08-27 00:00:00+08:00 至 2025-11-03 00:00:00+08:00

  [1.5] 数据完整性检查...
    ✅ 所有必要数据已就绪

✓ 数据提取完成


In [10]:
# ============================================================
# 步骤2.1：计算豆棕利润价差
# ============================================================
print("\n[3.1/9] 计算豆棕利润价差...")
logger.info("\n" + "="*60)
logger.info("步骤2.1：豆棕利润价差")
logger.info("="*60)

# 检查焦煤和焦炭数据是否存在
if 'P' in price_data and 'Y' in price_data:
    print("  [2.1.1] 配置价差参数...")

    
    # 注册豆棕价差配置
    spread_calc.create_spread(
        spread_name='SoybeanOil_PalmOil_spread',
        components={
            'Y': 1.0,        # 豆油价格系数
            'P': -1  # 焦煤价格系数（负数表示成本）
        }.items(),
        save_config=True
    )
    
    print("  [2.1.2] 计算价差...")
    # 计算价差
    sp_spread_df = spread_calc.calculate_spread(
        'SoybeanOil_PalmOil_spread',
        price_data,
        price_column='close'
    )
    
    # 添加统计特征
    sp_spread_df = spread_calc.get_spread_statistics(sp_spread_df, window=20)
    spread_data['SoybeanOil_PalmOil_spread'] = sp_spread_df
    
    print(f"    ✓ 价差数据点: {len(sp_spread_df)}")
    print(f"    ✓ 时间范围: {sp_spread_df.index.min()} 至 {sp_spread_df.index.max()}")
    
    # 价差统计信息
    print(f"\n  [2.1.3] 价差统计:")
    print(f"    均值: {sp_spread_df['spread'].mean():.2f} 元/吨")
    print(f"    标准差: {sp_spread_df['spread'].std():.2f} 元/吨")
    print(f"    最大值: {sp_spread_df['spread'].max():.2f} 元/吨")
    print(f"    最小值: {sp_spread_df['spread'].min():.2f} 元/吨")
    print(f"    当前值: {sp_spread_df['spread'].iloc[-1]:.2f} 元/吨")
    
    # 分析盈亏情况
    profitable_days = (sp_spread_df['spread'] > 0).sum()
    total_days = len(sp_spread_df)
    profit_ratio = profitable_days / total_days * 100
    
    
    if sp_spread_df['spread'].iloc[-1] > 0:
        print(f"    当前状态: 💰 盈利 {sp_spread_df['spread'].iloc[-1]:.2f} 元/吨")
    else:
        print(f"    当前状态: 📉 亏损 {abs(sp_spread_df['spread'].iloc[-1]):.2f} 元/吨")
    
    # 保存到数据库
    print("\n  [2.1.5] 保存到数据库...")
    db.insert_indicator_data(
        'SoybeanOil_PalmOil_spread',
        sp_spread_df[['spread']],
        metadata={
            'type': 'SoybeanOil_PalmOil_spread',
            'ratio': f'1:-1',
            'description': f'豆棕价差',
            'unit': '元/吨'
        }
    )
    print("    ✓ 数据已保存")
    
    # 平稳性检验
    print("\n  [2.1.6] 进行ADF平稳性检验...")
    indicator_builder.test_stationarity(
        sp_spread_df['spread'],
        name='SoybeanOil_PalmOil_spread'
    )
    
    logger.info(f"豆棕价差计算完成: {len(sp_spread_df)} 个数据点")
    
else:
    print("  ✗ 缺少豆油（Y）或棕榈油（P）价格数据")
    if 'Y' not in price_data:
        print("    缺少: 豆油（Y）")
    if 'P' not in price_data:
        print("    缺少: 棕榈油（P）")
    logger.error("缺少计算豆油棕榈油价差所需的价格数据")

print("\n✓ 豆油棕榈油价差计算完成")
logger.info("豆油棕榈油价差计算完成\n")

INFO:__main__:
INFO:__main__:步骤2.1：豆棕利润价差
INFO:__main__:============================================================
ERROR:src.core.spread_calculator:保存价差配置失败: Object of type dict_items is not JSON serializable
INFO:src.core.spread_calculator:创建价差配置: SoybeanOil_PalmOil_spread - 多头(1.0xY) - 空头(1.0xP)
INFO:src.core.spread_calculator:计算价差 SoybeanOil_PalmOil_spread: 4376 个数据点



[3.1/9] 计算豆棕利润价差...
  [2.1.1] 配置价差参数...
  [2.1.2] 计算价差...


INFO:src.core.spread_calculator:价差统计特征计算完成，窗口: 20


    ✓ 价差数据点: 4376
    ✓ 时间范围: 2007-10-29 00:00:00+08:00 至 2025-11-03 00:00:00+08:00

  [2.1.3] 价差统计:
    均值: 765.13 元/吨
    标准差: 628.00 元/吨
    最大值: 2304.00 元/吨
    最小值: -2428.00 元/吨
    当前值: -554.00 元/吨
    当前状态: 📉 亏损 554.00 元/吨

  [2.1.5] 保存到数据库...


INFO:src.core.database:插入了 0 条新指标数据
INFO:src.core.indicators:
INFO:src.core.indicators:ADF平稳性检验结果 - SoybeanOil_PalmOil_spread
INFO:src.core.indicators:==================================================
INFO:src.core.indicators:ADF统计量: -3.121815
INFO:src.core.indicators:P值: 0.024980
INFO:src.core.indicators:使用滞后阶数: 12
INFO:src.core.indicators:观测值数量: 4363
INFO:src.core.indicators:临界值:
INFO:src.core.indicators:  1%: -3.431850
INFO:src.core.indicators:  5%: -2.862203
INFO:src.core.indicators:  10%: -2.567123
INFO:src.core.indicators:结论: SoybeanOil_PalmOil_spread 是平稳序列 (p < 0.05)
INFO:src.core.indicators:==================================================

INFO:__main__:豆棕价差计算完成: 4376 个数据点
INFO:__main__:豆油棕榈油价差计算完成



    ✓ 数据已保存

  [2.1.6] 进行ADF平稳性检验...

✓ 豆油棕榈油价差计算完成


### 📈 可视化豆棕价差

查看盘面利润的历史变化趋势

In [11]:
# 可视化焦煤焦炭价差
if 'SoybeanOil_PalmOil_spread' in spread_data:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    
    sp_spread = spread_data['SoybeanOil_PalmOil_spread']
    
    # 创建图表
    fig, axes = plt.subplots(3, 1, figsize=(15, 12))
    
    # 子图1：价差时间序列
    ax1 = axes[0]
    ax1.plot(sp_spread.index, sp_spread['spread'], linewidth=1, color='navy', alpha=0.7)
    ax1.axhline(y=0, color='red', linestyle='--', linewidth=1, label='盈亏平衡线')
    ax1.fill_between(sp_spread.index, 0, sp_spread['spread'], 
                      where=(sp_spread['spread'] > 0), color='green', alpha=0.2, label='盈利区域')
    ax1.fill_between(sp_spread.index, 0, sp_spread['spread'], 
                      where=(sp_spread['spread'] <= 0), color='red', alpha=0.2, label='亏损区域')
    ax1.set_title('豆棕盘面价差1：1', fontsize=14, fontweight='bold')
    ax1.set_ylabel(' (元/吨)', fontsize=12)
    ax1.grid(True, alpha=0.3)
    ax1.legend(loc='best')
    
    # 子图2：价差分布直方图
    ax2 = axes[1]
    ax2.hist(sp_spread['spread'], bins=50, color='steelblue', alpha=0.7, edgecolor='black')
    ax2.axvline(x=0, color='red', linestyle='--', linewidth=2, label='盈亏平衡点')
    ax2.axvline(x=sp_spread['spread'].mean(), color='green', linestyle='--', linewidth=2, 
                label=f'均值: {sp_spread["spread"].mean():.2f}')
    ax2.set_title('盘面价差分布', fontsize=14, fontweight='bold')
    ax2.set_xlabel('利润 (元/吨)', fontsize=12)
    ax2.set_ylabel('频数', fontsize=12)
    ax2.legend(loc='best')
    ax2.grid(True, alpha=0.3, axis='y')
    
    # 子图3：滚动统计
    ax3 = axes[2]
    # 计算滚动均值和标准差
    rolling_mean = sp_spread['spread'].rolling(window=60).mean()
    rolling_std = sp_spread['spread'].rolling(window=60).std()

    ax3.plot(sp_spread.index, sp_spread['spread'], linewidth=1, color='lightgray', alpha=0.5, label='原始价差')
    ax3.plot(rolling_mean.index, rolling_mean, linewidth=2, color='blue', label='60日均值')
    ax3.fill_between(rolling_mean.index, 
                      rolling_mean - 2*rolling_std, 
                      rolling_mean + 2*rolling_std,
                      alpha=0.2, color='blue', label='±2标准差区间')
    ax3.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax3.set_title('60日滚动均值与波动区间', fontsize=14, fontweight='bold')
    ax3.set_xlabel('日期', fontsize=12)
    ax3.set_ylabel('利润 (元/吨)', fontsize=12)
    ax3.grid(True, alpha=0.3)
    ax3.legend(loc='best')
    
    plt.tight_layout()
    
    # 保存图表
    output_path = r'outputs\Palm&Soybean_Results\SoybeanOil_PalmOil_Spread_Analysis.png'
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f"✓ 图表已保存: {output_path}")
    
 
    
    # 打印关键统计信息
    print("\n" + "="*60)
    print("豆棕盘面价差统计摘要")
    print("="*60)
    print(f"配比系数: 1×豆油 - 1×棕榈油")
    print(f"数据周期: {sp_spread.index.min().strftime('%Y-%m-%d')} 至 {sp_spread.index.max().strftime('%Y-%m-%d')}")
    print(f"\n利润统计:")
    print(f"  平均利润: {sp_spread['spread'].mean():.2f} 元/吨")
    print(f"  中位数:   {sp_spread['spread'].median():.2f} 元/吨")
    print(f"  标准差:   {sp_spread['spread'].std():.2f} 元/吨")
    print(f"  最大利润: {sp_spread['spread'].max():.2f} 元/吨 ({sp_spread['spread'].idxmax().strftime('%Y-%m-%d')})")
    print(f"  最小利润: {sp_spread['spread'].min():.2f} 元/吨 ({sp_spread['spread'].idxmin().strftime('%Y-%m-%d')})")
    print(f"  当前利润: {sp_spread['spread'].iloc[-1]:.2f} 元/吨")

    print("="*60)
else:
    print("⚠️  未找到 COKING_PROFIT 价差数据，请先运行价差计算单元格")

✓ 图表已保存: outputs\Palm&Soybean_Results\SoybeanOil_PalmOil_Spread_Analysis.png

豆棕盘面价差统计摘要
配比系数: 1×豆油 - 1×棕榈油
数据周期: 2007-10-29 至 2025-11-03

利润统计:
  平均利润: 765.13 元/吨
  中位数:   836.00 元/吨
  标准差:   628.00 元/吨
  最大利润: 2304.00 元/吨 (2012-10-17)
  最小利润: -2428.00 元/吨 (2024-12-04)
  当前利润: -554.00 元/吨


In [12]:
# ============================================================
# 步骤3：特征工程
# ============================================================
print("\n[4/9] 开始特征工程...")
logger.info("\n" + "="*60)
logger.info("步骤3：特征工程")
logger.info("="*60)

# 获取主价差数据
main_spread = spread_data.get('SoybeanOil_PalmOil_spread')
if main_spread is None or main_spread.empty:
    print("  ✗ 价差数据不可用，停止执行")
    logger.error("价差数据不可用")
    raise ValueError("价差数据不可用")

# 1. 创建价差特征
print("  [3.1] 创建价差特征...")
spread_features = feature_engineer.create_spread_features(main_spread)
print(f"    ✓ 价差特征: {len(spread_features.columns)} 个")

# 2. 创建价格特征
print("  [3.2] 创建价格特征...")
price_features = feature_engineer.create_price_features(price_data)
print(f"    ✓ 价格特征: {len(price_features.columns)} 个")

# 3. 创建技术指标特征
print("  [3.3] 创建技术指标特征...")
technical_features = pd.DataFrame(index=main_spread.index)
for symbol, df in price_data.items():
    if symbol in [np.nan]:
        tech_df = feature_engineer.create_technical_features(df, symbol)
        # 选择关键列
        key_cols = [col for col in tech_df.columns 
                   if any(x in col for x in ['RSI', 'MACD', 'BB_percent'])]
        if key_cols:
            technical_features = technical_features.join(tech_df[key_cols], how='outer')
print(f"    ✓ 技术指标特征: {len(technical_features.columns)} 个")

INFO:__main__:
INFO:__main__:步骤3：特征工程
INFO:__main__:============================================================
INFO:src.core.feature_engineering:创建价差特征完成，特征数: 39
INFO:src.core.feature_engineering:创建价格特征完成，特征数: 32



[4/9] 开始特征工程...
  [3.1] 创建价差特征...
    ✓ 价差特征: 51 个
  [3.2] 创建价格特征...
    ✓ 价格特征: 32 个
  [3.3] 创建技术指标特征...
    ✓ 技术指标特征: 0 个


In [13]:

print("  [3.5] 创建宏观特征...")

temp_macro = main_spread.reset_index()
temp_macro.columns = ['date'] + list(main_spread.columns)

# 确保date列是datetime类型且无时区
temp_macro['date'] = pd.to_datetime(temp_macro['date'],utc=True)
if hasattr(temp_macro['date'].dtype, 'tz') and temp_macro['date'].dtype.tz is not None:
    temp_macro['date'] = temp_macro['date'].dt.tz_localize(None)

print(f"主数据时间类型: {temp_macro['date'].dtype}, 前5行:")
print(temp_macro[['date']].head())

for symbol in ['VIX', 'DXY', 'A','B']:
    df = db.get_price_data(symbol)
    if not df.empty and 'close' in df.columns:
        # 准备宏观数据：重置索引，计算收益率
        macro_df = df[['close']].copy()
        macro_df[f'{symbol}_return_1d'] = macro_df['close'].pct_change()
        macro_df = macro_df.reset_index()
        macro_df.columns = ['date', f'{symbol}_close', f'{symbol}_return_1d']
        
        # 确保有干净的datetime列（不带时区）
        # 如果原数据带时区，先用utc=True转换，再移除时区
        macro_df['date'] = pd.to_datetime(macro_df['date'], utc=True)
        if hasattr(macro_df['date'].dtype, 'tz') and macro_df['date'].dtype.tz is not None:
            macro_df['date'] = macro_df['date'].dt.tz_localize(None)
        
        print(f"{symbol}数据时间类型: {macro_df['date'].dtype}")
        
        # 使用merge_asof进行时间对齐（向后填充）
        temp_macro = pd.merge_asof(
            temp_macro.sort_values('date'),
            macro_df[['date', f'{symbol}_close', f'{symbol}_return_1d']].sort_values('date'),
            on='date',
            direction='backward'  # 使用最近的历史数据
        )
        
        # 显示对齐效果
        aligned_count = temp_macro[f'{symbol}_close'].notna().sum()
        missing_count = temp_macro[f'{symbol}_close'].isnull().sum()
        print(f"✓ {symbol}: {len(df)}条原始 → {aligned_count}条对齐（缺失{missing_count}个）")

# 恢复为索引格式（确保索引无时区）
macro_features = temp_macro.set_index('date')
# 再次确认索引无时区
# if hasattr(macro_features.index.dtype, 'tz') and macro_features.index.tz is not None:
#     macro_features.index = macro_features.index.tz_localize(None)

# 只保留宏观数据列，去除来自main_spread的列（避免与spread_features重复）
macro_cols = [col for col in macro_features.columns if any(x in col for x in ['VIX', 'DXY', 'RB'])]
macro_features = macro_features[macro_cols]

print("\n宏观特征前20行:")
print(macro_features.head(20))
print(f"宏观特征索引类型: {macro_features.index.dtype}")

print(f"    ✓ 宏观特征: {len(macro_features.columns)} 个（已时间对齐）")

  [3.5] 创建宏观特征...
主数据时间类型: datetime64[ns], 前5行:
                 date
0 2007-10-28 16:00:00
1 2007-10-29 16:00:00
2 2007-10-30 16:00:00
3 2007-10-31 16:00:00
4 2007-11-01 16:00:00
VIX数据时间类型: datetime64[ns]
✓ VIX: 6498条原始 → 4376条对齐（缺失0个）
DXY数据时间类型: datetime64[ns]
✓ DXY: 6528条原始 → 4376条对齐（缺失0个）
A数据时间类型: datetime64[ns]
✓ A: 5067条原始 → 4376条对齐（缺失0个）
B数据时间类型: datetime64[ns]
✓ B: 4272条原始 → 4376条对齐（缺失0个）

宏观特征前20行:
                     VIX_close  VIX_return_1d  DXY_close  DXY_return_1d
date                                                                   
2007-10-28 16:00:00  19.559999      -0.076051  77.029999      -0.003235
2007-10-29 16:00:00  19.870001       0.015849  76.839996      -0.002467
2007-10-30 16:00:00  21.070000       0.060392  76.769997      -0.000911
2007-10-31 16:00:00  18.530001      -0.120551  76.480003      -0.003777
2007-11-01 16:00:00  23.209999       0.252563  76.589996       0.001438
2007-11-04 16:00:00  23.010000      -0.008617  76.339996      -0.003264
2007-11-05 16

In [14]:
fundamental_data["INVENTORY"] = pd.merge(fundamental_data['SOYBEAN_INVENTORY'],
         fundamental_data['PALM_OIL_INVENTORY'],
            left_index=True,
            right_index=True,
            how='outer').ffill().dropna()

In [15]:

# 6. 创建基本面特征（豆油棕榈油库存数据）
print("  [3.6] 创建基本面特征...")

if "INVENTORY" in fundamental_data and\
        not fundamental_data['INVENTORY'].empty:
    # 获取基本面数据
    inventory = fundamental_data['INVENTORY'].copy()

    # 重置索引以便使用merge_asof
    inventory_reset = inventory.reset_index()
    inventory_reset.columns = ['date'] + list(inventory.columns)

    # 统一时区处理：转换为UTC再移除时区
    inventory_reset['date'] = pd.to_datetime(inventory_reset['date'], utc=True)
    if hasattr(inventory_reset['date'].dtype, 'tz') and inventory_reset['date'].dtype.tz is not None:
        inventory_reset['date'] = inventory_reset['date'].dt.tz_localize(None)

    print(f"    基本面数据时间类型: {inventory_reset['date'].dtype}")

    # 使用主数据的临时DataFrame（temp_macro已经准备好）
    # 使用merge_asof进行时间对齐
    temp_fundamental = temp_macro[['date']].copy()
    
    # 对齐基本面数据
    temp_fundamental = pd.merge_asof(
        temp_fundamental.sort_values('date'),
        inventory_reset.sort_values('date'),
        on='date',
        direction='backward'  # 使用最近的历史数据
    )
    
    # 计算基本面数据的变化率
    for col in inventory.columns:
        if col in temp_fundamental.columns:
            # 计算变化率
            temp_fundamental[f'{col}_pct_change'] = temp_fundamental[col].pct_change()
            # 计算滚动均值
            temp_fundamental[f'{col}_ma20'] = temp_fundamental[col].rolling(20).mean()
    
    # 恢复为索引格式
    fundamental_features = temp_fundamental.set_index('date')
    
    # 只保留基本面相关列
    fundamental_cols = [col for col in fundamental_features.columns 
                       if any(x in col for x in [
                           'warehouse_receipt', 'inventory', 'inflow',
                           'pct_change', 'ma20'
                       ])]
    fundamental_features = fundamental_features[fundamental_cols]
    
    # 显示对齐效果
    print(f"    ✓ 基本面特征: {len(fundamental_features.columns)} 个")
    aligned_count = fundamental_features.notna().any(axis=1).sum()
    print(f"    ✓ 时间对齐: {aligned_count} 条有效数据")
    
    print("\n    基本面特征列表:")
    for col in fundamental_features.columns:
        non_null = fundamental_features[col].notna().sum()
        print(f"      - {col}: {non_null} 个有效值")
    
else:
    print("    ⚠️  无 INVENTORY 基本面数据")
    fundamental_features = pd.DataFrame(index=main_spread.index)
    print("    ✓ 创建空的基本面特征DataFrame")


  [3.6] 创建基本面特征...
    基本面数据时间类型: datetime64[ns]
    ✓ 基本面特征: 9 个


    ✓ 时间对齐: 3024 条有效数据

    基本面特征列表:
      - soybean_port_inventory: 3024 个有效值
      - soybean_oil_port_inventory: 3024 个有效值
      - palm_oil_port_inventory: 3024 个有效值
      - soybean_port_inventory_pct_change: 3023 个有效值
      - soybean_port_inventory_ma20: 3005 个有效值
      - soybean_oil_port_inventory_pct_change: 3023 个有效值
      - soybean_oil_port_inventory_ma20: 3005 个有效值
      - palm_oil_port_inventory_pct_change: 3014 个有效值
      - palm_oil_port_inventory_ma20: 3005 个有效值


In [16]:



# 6. 创建目标变量
print("  [3.6] 创建目标变量...")
target_df = feature_engineer.create_target_variable(
    main_spread,
    method='sharpe_regress',
    forward_period=10,
)
print(f"    ✓ 目标变量创建完成")

# 🔍 调试点3：在此处设置断点，检查各个特征DataFrame
# 可以查看: spread_features.head(), price_features.shape, technical_features.columns

# ============================================================
# 合并特征
# ============================================================
print("\n  [3.7] 合并所有特征...")
logger.info("合并特征...")

all_features_list = [
    spread_features,
    price_features,
    technical_features,
    fundamental_features,
    macro_features,
    target_df[['target']]
]
# print(pd.concat(all_features_list, axis=1).head(20))

# 诊断：打印合并前的状态
print("\n  诊断信息 - 合并前各DataFrame状态:")

df_names = ['价差特征', '价格特征', '技术指标', '季节性特征', '宏观特征', '目标变量']

# 统一清理所有DataFrame的索引时区
print("\n  [3.7.1] 统一清理索引时区...")
for i, (name, df) in enumerate(zip(df_names, all_features_list)):
    # 检查索引是否有时区
    df.index = pd.to_datetime(df.index, utc=True)
    if hasattr(df.index, 'tz') and df.index.tz is not None:
        print(f"    {name}: 移除时区 {df.index.tz}")
        all_features_list[i].index = df.index.tz_localize(None)
    
    # 打印诊断信息
    null_count = df.isnull().sum().sum()
    index_type = type(df.index).__name__
    index_dtype = df.index.dtype if hasattr(df.index, 'dtype') else 'N/A'
    print(f"    {name}: {len(df)}样本, {len(df.columns)}列, {null_count}缺失值, 索引类型:{index_type}({index_dtype})")
    if null_count > 0:
        null_cols = df.isnull().sum()
        null_cols = null_cols[null_cols > 0]
        print(f"      缺失值列: {dict(list(null_cols.items())[:3])}")

# 合并特征
print("\n  [3.7.2] 开始合并特征...")
features_df = feature_engineer.merge_all_features(all_features_list)

print(f"\n  合并后状态:")
print(f"    总样本数: {len(features_df)}")
print(f"    总特征数: {len(features_df.columns)}")
print(f"    总缺失值: {features_df.isnull().sum().sum()}")

# ============================================================
# 清理缺失值
# ============================================================
print("\n  [3.8] 清理缺失值...")

# 显示缺失值最多的列
total_nulls = features_df.isnull().sum().sum()
if total_nulls > 0:
    null_counts = features_df.isnull().sum()
    cols_with_nulls = null_counts[null_counts > 0].sort_values(ascending=False)
    print(f"    缺失值最多的前5列:")
    for col, count in cols_with_nulls.head(5).items():
        pct = count / len(features_df) * 100
        print(f"      {col}: {count} ({pct:.2f}%)")

# 步骤1：前向填充
print("\n    步骤1: ffill前向填充...")
exclude_target = features_df.columns.difference(['target', 'forward_return'])
features_df[exclude_target] = features_df[exclude_target].fillna(method='ffill')
remaining_nulls = features_df.isnull().sum().sum()
print(f"      剩余缺失值: {remaining_nulls}")



print(f"\n  最终清理结果:")
print(f"    剩余样本数: {len(features_df)}")
print(f"    剩余特征数: {len(features_df.columns) - 2}")  # 减去target和forward_return
print(f"    缺失值: {features_df.isnull().sum().sum()}")

print("✓ 特征工程完成")
logger.info(f"特征构建完成，总特征数: {len(features_df.columns) - 2}")
logger.info(f"样本数: {len(features_df)}")

# 🔍 调试点4：在此处设置断点，检查 features_df
# 可以使用: features_df.describe(), features_df.head(), features_df.isnull().sum()

INFO:src.core.feature_engineering:创建夏普比率回归目标变量
INFO:__main__:合并特征...
INFO:src.core.feature_engineering:特征合并完成，总特征数: 97, 样本数: 4376


  [3.6] 创建目标变量...
    ✓ 目标变量创建完成

  [3.7] 合并所有特征...

  诊断信息 - 合并前各DataFrame状态:

  [3.7.1] 统一清理索引时区...
    价差特征: 移除时区 UTC
    价差特征: 4376样本, 51列, 932缺失值, 索引类型:DatetimeIndex(datetime64[ns])
      缺失值列: {'spread_mean': 19, 'spread_std': 19, 'spread_min': 19}
    价格特征: 移除时区 UTC
    价格特征: 4818样本, 32列, 7604缺失值, 索引类型:DatetimeIndex(datetime64[ns])
      缺失值列: {'P_return_1d': 438, 'P_return_5d': 442, 'P_return_10d': 447}
    技术指标: 移除时区 UTC
    技术指标: 4376样本, 0列, 0.0缺失值, 索引类型:DatetimeIndex(datetime64[ns])
    季节性特征: 移除时区 UTC
    季节性特征: 4376样本, 9列, 12237缺失值, 索引类型:DatetimeIndex(datetime64[ns])
      缺失值列: {'soybean_port_inventory': 1352, 'soybean_oil_port_inventory': 1352, 'palm_oil_port_inventory': 1352}
    宏观特征: 移除时区 UTC
    宏观特征: 4376样本, 4列, 0缺失值, 索引类型:DatetimeIndex(datetime64[ns])
    目标变量: 移除时区 UTC
    目标变量: 4376样本, 1列, 10缺失值, 索引类型:DatetimeIndex(datetime64[ns])
      缺失值列: {'target': 10}

  [3.7.2] 开始合并特征...

  合并后状态:
    总样本数: 4376
    总特征数: 97
    总缺失值: 13445

  [3.8] 清理缺失值...
    缺失值最多的前5列:

INFO:__main__:特征构建完成，总特征数: 95
INFO:__main__:样本数: 4376


      剩余缺失值: 13436

  最终清理结果:
    剩余样本数: 4376
    剩余特征数: 95
    缺失值: 13436
✓ 特征工程完成


In [17]:
# ============================================================
# 测试：查看基本面数据和特征
# ============================================================
print("📊 查看豆棕基本面数据")
print("="*60)

if 'INVENTORY' in fundamental_data and not fundamental_data['INVENTORY'].empty:
    inventory = fundamental_data['INVENTORY']

    print(f"\n数据基本信息:")
    print(f"  样本数: {len(inventory)}")
    print(f"  时间范围: {inventory.index.min()} 至 {inventory.index.max()}")
    print(f"  时区: {inventory.index.tz}")
    
    print(f"\n数据字段:")
    for col in inventory.columns:
        print(f"  - {col}")
    
    print(f"\n数据统计:")
    print(inventory.describe())

    print(f"\n最新数据（最近5条）:")
    print(inventory.tail(5))

    print(f"\n数据缺失情况:")
    missing = inventory.isnull().sum()
    for col, count in missing.items():
        pct = count / len(inventory) * 100
        print(f"  {col}: {count} ({pct:.1f}%)")
    
else:
    print("⚠️  未找到 INVENTORY 数据")
    print("请先运行数据导入单元格")

print("\n" + "="*60)

📊 查看豆棕基本面数据

数据基本信息:
  样本数: 3436
  时间范围: 2013-05-14 00:00:00+08:00 至 2025-11-03 00:00:00+08:00
  时区: Asia/Shanghai

数据字段:
  - soybean_port_inventory
  - soybean_oil_port_inventory
  - palm_oil_port_inventory

数据统计:
       soybean_port_inventory  soybean_oil_port_inventory  \
count             3436.000000                 3436.000000   
mean               662.540681                   95.302480   
std                 88.078624                   25.128321   
min                349.190000                   43.600000   
25%                619.615000                   76.500000   
50%                674.350000                   92.800000   
75%                702.572500                  115.750000   
max                859.080000                  164.750000   

       palm_oil_port_inventory  
count              3436.000000  
mean                 63.673588  
std                  26.674878  
min                   0.000000  
25%                  43.600000  
50%                  58.310000  
75% 

In [18]:
features_df = features_df.dropna()

In [19]:
import importlib
import src.core.ml_models
importlib.reload(src.core.ml_models)
from src.core.ml_models import MLModel,SignalGenerator

In [20]:
# ============================================================
# 步骤4：模型训练
# ============================================================
print("\n[5/9] 开始模型训练...")
logger.info("\n" + "="*60)
logger.info("步骤4：模型训练")
logger.info("="*60)

if features_df is None or features_df.empty:
    print("  ✗ 特征数据不可用，停止执行")
    logger.error("特征数据不可用")
    raise ValueError("特征数据不可用")

# 创建模型
print("  [4.1] 创建模型...")
model = MLModel(model_type='gradient_boosting', task='regression')
print("    ✓ 使用 Gradient Boosting 分类器")

# 准备数据 - 先划分训练集和测试集（避免数据泄露）
print("  [4.2] 准备训练/测试数据...")
X_train, X_test, y_train, y_test, train_idx, test_idx = model.prepare_data(
    features_df,
    target_col='target',
    feature_cols=None,  # 暂时使用所有特征
    test_size=0.2,
    scale=False  # 暂时不标准化，特征选择后再标准化
)
print(f"    ✓ 训练集: {len(X_train)} 样本")
print(f"    ✓ 测试集: {len(X_test)} 样本")

# 特征选择 - 仅在训练集上进行（避免数据泄露）
print("  [4.3] 特征选择（仅基于训练集）...")
# 重建训练集DataFrame用于特征选择
all_feature_cols = [col for col in features_df.columns if col != 'target']
train_df = pd.DataFrame(X_train, columns=all_feature_cols, index=train_idx)
train_df['target'] = y_train

selected_features = feature_engineer.select_features(
    train_df,
    target_col='target',
    method='variance',
    top_k=50
)
print(f"    ✓ 选择了 {len(selected_features)} 个特征")

# 使用选定的特征重新准备数据
print("  [4.4] 使用选定特征重新准备数据...")
X_train_selected, X_test_selected, y_train, y_test, train_idx, test_idx = model.prepare_data(
    features_df,
    target_col='target',
    feature_cols=selected_features,
    test_size=0.2,
    scale=True  # 现在进行标准化
)
print(f"    ✓ 训练集: {len(X_train_selected)} 样本, {X_train_selected.shape[1]} 个特征")
print(f"    ✓ 测试集: {len(X_test_selected)} 样本, {X_test_selected.shape[1]} 个特征")

# 训练模型
print("  [4.5] 训练模型...")
model_params = {
    'n_estimators': 500,
    'max_depth': 8,
    'learning_rate': 0.1,
    'random_state': 42
}
model.train(X_train_selected, y_train, **model_params)
print("    ✓ 模型训练完成")

# 评估模型
print("  [4.6] 评估模型...")
metrics = model.evaluate(X_test_selected, y_test)
print(f"    ✓ 准确率: {metrics.get('accuracy', 0):.4f}")
print(f"    ✓ F1分数: {metrics.get('f1', 0):.4f}")

# 可视化特征重要性
if model.feature_importance is not None:
    print("  [4.7] 生成特征重要性图...")
    visualizer.plot_feature_importance(model.feature_importance, top_n=15)
    print("    ✓ 特征重要性图已保存")

# 保存模型
print("  [4.8] 保存模型...")
model_path = Path('models/coking_model.pkl')
model_path.parent.mkdir(exist_ok=True)
model.save_model(str(model_path))
print(f"    ✓ 模型已保存到: {model_path}")

print("✓ 模型训练完成")
logger.info("模型训练完成\n")

# 🔍 调试点5：在此处设置断点，检查模型和训练结果
# 可以查看: model.feature_importance, metrics, X_train_selected.shape, X_test_selected.shape

# ============================================================
# 步骤5：回测
# ============================================================
print("\n[6/9] 开始回测...")
logger.info("\n" + "="*60)
logger.info("步骤5：回测")
logger.info("="*60)


INFO:__main__:
INFO:__main__:步骤4：模型训练
INFO:__main__:============================================================
INFO:src.core.ml_models:数据准备完成:
INFO:src.core.ml_models:  特征数: 96
INFO:src.core.ml_models:  训练集样本数: 2396
INFO:src.core.ml_models:  测试集样本数: 599
INFO:src.core.feature_engineering:特征选择完成，选择了 50 个特征
INFO:src.core.ml_models:数据准备完成:
INFO:src.core.ml_models:  特征数: 50
INFO:src.core.ml_models:  训练集样本数: 2396
INFO:src.core.ml_models:  测试集样本数: 599
INFO:src.core.ml_models:开始训练 gradient_boosting 模型...



[5/9] 开始模型训练...
  [4.1] 创建模型...
    ✓ 使用 Gradient Boosting 分类器
  [4.2] 准备训练/测试数据...
    ✓ 训练集: 2396 样本
    ✓ 测试集: 599 样本
  [4.3] 特征选择（仅基于训练集）...
    ✓ 选择了 50 个特征
  [4.4] 使用选定特征重新准备数据...
    ✓ 训练集: 2396 样本, 50 个特征
    ✓ 测试集: 599 样本, 50 个特征
  [4.5] 训练模型...


INFO:src.core.ml_models:Top 10 重要特征:
INFO:src.core.ml_models:                         feature  importance
7                        Y_ma_10    0.131953
38                spread_diff_1d    0.071915
22                 spread_max_10    0.048964
44  palm_oil_port_inventory_ma20    0.037964
9                        Y_ma_50    0.037405
34   soybean_port_inventory_ma20    0.036057
4                        P_ma_20    0.035662
1                 Y_volume_ma_20    0.034097
40                   spread_macd    0.033626
5                        P_ma_50    0.029069
INFO:src.core.ml_models:模型训练完成
INFO:src.core.ml_models:
模型评估结果:
INFO:src.core.ml_models:MSE: 8.838518
INFO:src.core.ml_models:RMSE: 2.972965
INFO:src.core.ml_models:MAE: 2.351336
INFO:src.core.ml_models:R²: -0.4073


    ✓ 模型训练完成
  [4.6] 评估模型...
    ✓ 准确率: 0.0000
    ✓ F1分数: 0.0000
  [4.7] 生成特征重要性图...


INFO:src.core.visualization:特征重要性图表已保存: outputs/Palm&Soybean_Results/feature_importance.png
INFO:src.core.ml_models:模型已保存: models\coking_model.pkl
INFO:__main__:模型训练完成

INFO:__main__:
INFO:__main__:步骤5：回测
INFO:__main__:============================================================


    ✓ 特征重要性图已保存
  [4.8] 保存模型...
    ✓ 模型已保存到: models\coking_model.pkl
✓ 模型训练完成

[6/9] 开始回测...


In [ ]:
# # ============================================================
# # 步骤4：模型训练
# # ============================================================
# print("\n[5/9] 开始模型训练...")
# logger.info("\n" + "="*60)
# logger.info("步骤4：模型训练")
# logger.info("="*60)

# if features_df is None or features_df.empty:
#     print("  ✗ 特征数据不可用，停止执行")
#     logger.error("特征数据不可用")
#     raise ValueError("特征数据不可用")

# # 特征选择
# print("  [4.1] 特征选择...")
# selected_features = feature_engineer.select_features(
#     features_df,
#     target_col='target',
#     method='variance',
#     top_k=50
# )
# print(f"    ✓ 选择了 {len(selected_features)} 个特征")

# # 创建模型
# print("  [4.2] 创建模型...")
# model = MLModel(model_type='gradient_boosting', task='regression')
# print("    ✓ 使用 Gradient Boosting 分类器")

# # 准备数据
# print("  [4.3] 准备训练/测试数据...")
# X_train, X_test, y_train, y_test, train_idx, test_idx = model.prepare_data(
#     features_df,
#     target_col='target',
#     feature_cols=selected_features,
#     test_size=0.2,
#     scale=True
# )
# print(f"    ✓ 训练集: {len(X_train)} 样本")
# print(f"    ✓ 测试集: {len(X_test)} 样本")

# # 训练模型
# print("  [4.4] 训练模型...")
# model_params = {
#     'n_estimators': 500,
#     'max_depth': 8,
#     'learning_rate': 0.1,
#     'random_state': 42
# }
# model.train(X_train, y_train, **model_params)
# print("    ✓ 模型训练完成")

# # 评估模型
# print("  [4.5] 评估模型...")
# metrics = model.evaluate(X_test, y_test)
# # print(f"    ✓ 准确率: {metrics.get('accuracy', 0):.4f}")
# # print(f"    ✓ F1分数: {metrics.get('f1', 0):.4f}")

# # 可视化特征重要性
# if model.feature_importance is not None:
#     print("  [4.6] 生成特征重要性图...")
#     visualizer.plot_feature_importance(model.feature_importance, top_n=15)
#     print("    ✓ 特征重要性图已保存")

# # 保存模型
# print("  [4.7] 保存模型...")
# model_path = Path('models/sp_model.pkl')
# model_path.parent.mkdir(exist_ok=True)
# model.save_model(str(model_path))
# print(f"    ✓ 模型已保存到: {model_path}")

# print("✓ 模型训练完成")
# logger.info("模型训练完成\n")

# # 🔍 调试点5：在此处设置断点，检查模型和训练结果
# # 可以查看: model.feature_importance, metrics, X_train.shape, X_test.shape

# # ============================================================
# # 步骤5：回测
# # ============================================================
# print("\n[6/9] 开始回测...")
# logger.info("\n" + "="*60)
# logger.info("步骤5：回测")
# logger.info("="*60)

INFO:__main__:
INFO:__main__:步骤4：模型训练
INFO:__main__:============================================================
INFO:src.core.feature_engineering:特征选择完成，选择了 50 个特征
INFO:src.core.ml_models:数据准备完成:
INFO:src.core.ml_models:  特征数: 50
INFO:src.core.ml_models:  训练集样本数: 2395
INFO:src.core.ml_models:  测试集样本数: 599
INFO:src.core.ml_models:开始训练 gradient_boosting 模型...



[5/9] 开始模型训练...
  [4.1] 特征选择...
    ✓ 选择了 50 个特征
  [4.2] 创建模型...
    ✓ 使用 Gradient Boosting 分类器
  [4.3] 准备训练/测试数据...
    ✓ 训练集: 2395 样本
    ✓ 测试集: 599 样本
  [4.4] 训练模型...


INFO:src.core.ml_models:Top 10 重要特征:
INFO:src.core.ml_models:                        feature  importance
7                       Y_ma_10    0.123712
42               spread_diff_1d    0.061728
43                  spread_macd    0.051507
40  soybean_port_inventory_ma20    0.049224
49      palm_oil_port_inventory    0.046107
9                       Y_ma_50    0.045255
23                spread_max_10    0.035301
4                       P_ma_20    0.035293
22                spread_ema_26    0.032197
0                P_volume_ma_20    0.032141
INFO:src.core.ml_models:模型训练完成
INFO:src.core.ml_models:
模型评估结果:
INFO:src.core.ml_models:MSE: 7.588987
INFO:src.core.ml_models:RMSE: 2.754812
INFO:src.core.ml_models:MAE: 2.208384
INFO:src.core.ml_models:R²: -0.2108


    ✓ 模型训练完成
  [4.5] 评估模型...
    ✓ 准确率: 0.0000
    ✓ F1分数: 0.0000
  [4.6] 生成特征重要性图...


INFO:src.core.visualization:特征重要性图表已保存: outputs/Palm&Soybean_Results/feature_importance.png
INFO:src.core.ml_models:模型已保存: models\sp_model.pkl
INFO:__main__:模型训练完成

INFO:__main__:
INFO:__main__:步骤5：回测
INFO:__main__:============================================================


    ✓ 特征重要性图已保存
  [4.7] 保存模型...
    ✓ 模型已保存到: models\sp_model.pkl
✓ 模型训练完成

[6/9] 开始回测...


In [36]:
import importlib
import src.core.ml_models
importlib.reload(src.core.backtest)
from src.core.backtest import BacktestEngine

In [23]:
# 生成信号
print("  [5.1] 生成交易信号...")
signal_generator = SignalGenerator(model, 
                                use_rolling_quantile=True,      # ✅ 使用滚动分位数
                                rolling_window=250,              # 250天窗口
                                upper_quantile=0.8,            # 85%分位数
                                lower_quantile=0.2,            # 5%分位数
                                signal_holding_days=20          # 信号维持20天
                                   )
signals = signal_generator.generate_signals(X_test_selected)
signals.index = test_idx
print(f"    ✓ 生成 {len(signals)} 个信号")
print(f"    信号分布: {signals.value_counts().to_dict()}")

# 获取价差价格数据
print("  [5.2] 准备价格数据...")
# 确保spread_data索引与test_idx时区一致
spread_df_for_backtest = spread_data['SoybeanOil_PalmOil_spread'].copy()
spread_df_for_backtest.index = pd.to_datetime(spread_df_for_backtest.index,utc=True)
if hasattr(spread_df_for_backtest.index, 'tz') and spread_df_for_backtest.index.tz is not None:
    spread_df_for_backtest.index = spread_df_for_backtest.index.tz_localize(None)

spread_prices = spread_df_for_backtest.loc[test_idx, ['spread']].copy()
spread_prices.columns = ['close']
spread_prices['volatility'] = spread_prices['close'].pct_change().rolling(20).std()
print(f"    ✓ 价格数据: {len(spread_prices)} 条")


# 运行回测
print("  [5.3] 运行回测...")
backtest_engine = BacktestEngine(
    initial_capital=1000000,
    commission_rate=0.0005,
    slippage_rate=0.0001,
    max_position=100000,
    max_capital_usage=0.3,
    leverage=1.0,
    stop_loss_pct=0.0
)

equity_curve = backtest_engine.run_backtest(
    spread_prices,
    signals,
    price_col='close',
    volatility_col='volatility'
)
print(f"    ✓ 回测完成，最终权益: ${equity_curve['equity'].iloc[-1]:,.2f}")

# 获取交易日志
trade_log = backtest_engine.get_trade_log()
print(f"    ✓ 总交易次数: {len(trade_log)}")

# 绩效分析
print("  [5.4] 绩效分析...")
analyzer = PerformanceAnalyzer(
    equity_curve,
    initial_capital=1000000,
    risk_free_rate=0.02
)

performance_report = analyzer.generate_performance_report(trade_log)
print(f"    ✓ 总收益率: {performance_report.get('total_return', 0)*100:.2f}%")
print(f"    ✓ 夏普比率: {performance_report.get('sharpe_ratio', 0):.2f}")
print(f"    ✓ 最大回撤: {performance_report.get('max_drawdown', 0)*100:.2f}%")

print("✓ 回测完成")
logger.info("回测完成\n")

# 🔍 调试点6：在此处设置断点，检查回测结果
# 可以查看: equity_curve.tail(), trade_log.head(), performance_report

# ============================================================
# 步骤6：可视化
# ============================================================
print("\n[7/9] 开始可视化...")
logger.info("\n" + "="*60)
logger.info("步骤6：结果可视化")
logger.info("="*60)

print("  [6.1] 生成价格和价差图...")
visualizer.plot_price_and_spread(
    price_data,
    spread_data['SoybeanOil_PalmOil_spread'],
    title='SoybeanOil_PalmOil_spread 1:1'
)
print("    ✓ price_spread_chart.png")

print("  [6.2] 生成权益曲线图...")
visualizer.plot_equity_curve(equity_curve)
print("    ✓ equity_curve.png")

print("  [6.3] 生成收益率分布图...")
returns = equity_curve['equity'].pct_change().dropna()
visualizer.plot_returns_distribution(returns)
print("    ✓ returns_distribution.png")

print("  [6.4] 生成月度收益热力图...")
visualizer.plot_monthly_returns_heatmap(equity_curve)
print("    ✓ monthly_returns_heatmap.png")

print("  [6.5] 生成滚动指标图...")
visualizer.plot_rolling_metrics(equity_curve, window=60)
print("    ✓ rolling_metrics.png")

print("  [6.6] 生成交易分析图...")
visualizer.plot_trade_analysis(trade_log)
print("    ✓ trade_analysis.png")

print("✓ 可视化完成")
logger.info("可视化完成\n")

# ============================================================
# 完成
# ============================================================
print("\n" + "="*60)
print("策略执行完成！")
print("="*60)
print("\n生成的文件:")
print("  📁 data/trading_data.db          - 数据库")
print("  📁 models/Coking_Profit_model.pkl - 模型文件")
print("  📁 outputs/charts/*.png          - 图表文件")
print("  📁 logs/debug_strategy.log       - 日志文件")

print("\n可用的全局变量（用于调试）:")
print("  数据相关:")
print("    - price_data       : 期货价格数据字典")
print("    - spread_data      : 价差数据字典")
print("    - macro_data       : 宏观数据字典")
print("    - fundamental_data : 基本面数据字典")
print("\n  特征相关:")
print("    - spread_features  : 价差特征DataFrame")
print("    - price_features   : 价格特征DataFrame")
print("    - technical_features: 技术指标特征DataFrame")
print("    - seasonal_features: 季节性特征DataFrame")
print("    - macro_features   : 宏观特征DataFrame")
print("    - target_df        : 目标变量DataFrame")
print("    - features_df      : 合并后的完整特征DataFrame")
print("\n  模型相关:")
print("    - model            : 训练好的模型")
print("    - X_train, X_test  : 训练/测试特征")
print("    - y_train, y_test  : 训练/测试标签")
print("    - selected_features: 选择的特征列表")
print("\n  回测相关:")
print("    - signals          : 交易信号Series")
print("    - equity_curve     : 权益曲线DataFrame")
print("    - trade_log        : 交易日志DataFrame")
print("    - performance_report: 绩效报告字典")

print("\n💡 调试提示:")
print("  1. 在VS Code中打开此文件")
print("  2. 点击行号左侧设置断点（蓝点）")
print("  3. 按F5或点击'运行和调试'启动调试")
print("  4. 在'变量'面板查看所有变量的值")
print("  5. 在'调试控制台'输入变量名查看详细信息")
print("  例如: price_data.keys(), features_df.shape, model.feature_importance")

logger.info("\n" + "="*60)
logger.info("所有任务完成！")
logger.info("="*60)

# 🔍 最终调试点：程序结束前，所有变量都已计算完成
# 现在可以检查任何变量的最终状态

INFO:src.core.ml_models:使用滚动分位数模式: window=250, 上分位数=0.8, 下分位数=0.2
INFO:src.core.ml_models:滚动分位数统计:
INFO:src.core.ml_models:  上阈值范围: [0.1902, 2.1423], 均值: 1.5027
INFO:src.core.ml_models:  下阈值范围: [-1.3198, -0.1855], 均值: -0.7473
INFO:src.core.ml_models:回归信号统计:
INFO:src.core.ml_models:  预测值范围: [-3.0578, 10.5497]
INFO:src.core.ml_models:  做多信号(1): 102 (17.0%)
INFO:src.core.ml_models:  观望信号(0): 332 (55.4%)
INFO:src.core.ml_models:  做空信号(-1): 165 (27.5%)
INFO:src.core.ml_models:生成交易信号完成，信号分布:
INFO:src.core.ml_models: 0    332
-1    165
 1    102
Name: signal, dtype: int64
INFO:src.core.ml_models:应用20天信号维持后，信号分布:
INFO:src.core.ml_models:-1    331
 1    226
 0     42
Name: signal, dtype: int64
INFO:src.core.backtest:开始运行回测...
INFO:src.core.backtest:初始资金: $1,000,000.00
INFO:src.core.backtest:手续费率: 0.050%
INFO:src.core.backtest:滑点率: 0.010%


  [5.1] 生成交易信号...
    ✓ 生成 599 个信号
    信号分布: {(-3.0577608344194607, 1.969366077718638, -0.4342672784568817, -1, 1.0): 1, (0.6408634468964581, 2.0928252871289286, -0.681687658639559, 1, 0.0): 1, (0.6552094925130777, 0.21773233879633086, -1.2510277597571253, 1, 0.29785473757600495): 1, (0.6785836646979283, 2.0699595918427023, -0.6738910718822295, 1, 0.0): 1, (0.6792785807387388, 2.0334768686735005, -0.615002100473243, 0, 0.0): 1, (0.6917545858998811, 2.0334768686735005, -0.2603683102345741, -1, 0.0): 1, (0.7146986173276278, 1.9667994345388127, -0.615002100473243, 0, 0.0): 1, (0.7277892776815523, 2.0334768686735005, -0.615002100473243, 0, 0.0): 1, (0.7288211513928128, 2.0334768686735005, -0.3422881078527293, -1, 0.0): 1, (0.7293431968483458, 1.9667994345388127, -0.615002100473243, 0, 0.0): 1, (0.7525860828097727, 2.0649693006471748, -0.6638161003503107, 0, 0.0): 1, (0.7567564634443612, 2.0663220153204125, -0.6651451475643252, 0, 0.0): 1, (0.7635369628899812, 1.9905027522507674, -0.6150021

INFO:src.core.backtest:杠杆倍数: 1.0x
INFO:src.core.backtest:保证金比例: 10.0%
INFO:src.core.backtest:最大资金使用率: 30%
INFO:src.core.backtest:回测完成，共执行 533 笔交易
INFO:src.core.backtest:最终权益: $1,972,838.86
INFO:src.core.backtest:
INFO:src.core.backtest:绩效分析报告
INFO:src.core.backtest:============================================================
INFO:src.core.backtest:总收益率: 97.28%
INFO:src.core.backtest:年化收益率: 32.84%
INFO:src.core.backtest:年化波动率: 45.18%
INFO:src.core.backtest:夏普比率: 0.6826
INFO:src.core.backtest:索提诺比率: 1.0690
INFO:src.core.backtest:卡玛比率: 1.1218
INFO:src.core.backtest:最大回撤: -29.27%
INFO:src.core.backtest:VaR (95%): -1.43%
INFO:src.core.backtest:CVaR (95%): -4.87%
INFO:src.core.backtest:胜率: 54.92%
INFO:src.core.backtest:盈亏比: 1.54
INFO:src.core.backtest:总交易次数: 533
INFO:src.core.backtest:总手续费: $12,181.39
INFO:src.core.backtest:============================================================

INFO:__main__:回测完成

INFO:__main__:
INFO:__main__:步骤6：结果可视化
INFO:__main__:===================================

    ✓ 回测完成，最终权益: $1,972,838.86
    ✓ 总交易次数: 533
  [5.4] 绩效分析...
    ✓ 总收益率: 97.28%
    ✓ 夏普比率: 0.68
    ✓ 最大回撤: -29.27%
✓ 回测完成

[7/9] 开始可视化...
  [6.1] 生成价格和价差图...


INFO:src.core.visualization:价格和价差图表已保存: outputs/Palm&Soybean_Results/price_spread_chart.png


...

    ✓ monthly_returns_heatmap.png
  [6.5] 生成滚动指标图...


INFO:src.core.visualization:滚动指标图表已保存: outputs/Palm&Soybean_Results/rolling_metrics.png


    ✓ rolling_metrics.png
  [6.6] 生成交易分析图...


INFO:src.core.visualization:交易分析图表已保存: outputs/Palm&Soybean_Results/trade_analysis.png
INFO:__main__:可视化完成

INFO:__main__:
INFO:__main__:所有任务完成！
INFO:__main__:============================================================


    ✓ trade_analysis.png
✓ 可视化完成

策略执行完成！

生成的文件:
  📁 data/trading_data.db          - 数据库
  📁 models/Coking_Profit_model.pkl - 模型文件
  📁 outputs/charts/*.png          - 图表文件
  📁 logs/debug_strategy.log       - 日志文件

可用的全局变量（用于调试）:
  数据相关:
    - price_data       : 期货价格数据字典
    - spread_data      : 价差数据字典
    - macro_data       : 宏观数据字典
    - fundamental_data : 基本面数据字典

  特征相关:
    - spread_features  : 价差特征DataFrame
    - price_features   : 价格特征DataFrame
    - technical_features: 技术指标特征DataFrame
    - seasonal_features: 季节性特征DataFrame
    - macro_features   : 宏观特征DataFrame
    - target_df        : 目标变量DataFrame
    - features_df      : 合并后的完整特征DataFrame

  模型相关:
    - model            : 训练好的模型
    - X_train, X_test  : 训练/测试特征
    - y_train, y_test  : 训练/测试标签
    - selected_features: 选择的特征列表

  回测相关:
    - signals          : 交易信号Series
    - equity_curve     : 权益曲线DataFrame
    - trade_log        : 交易日志DataFrame
    - performance_report: 绩效报告字典

💡 调试提示:
  1. 在VS Code中打开此文件
  2. 点击行号左侧设置断点（

In [25]:
# ============================================================
# 步骤7：保存结果
# ============================================================
print("\n[8/9] 保存策略结果...")
logger.info("\n" + "="*60)
logger.info("步骤7：保存结果")
logger.info("="*60)

# 重新导入模块以获取最新代码
import importlib
import src.core.result_saver
importlib.reload(src.core.result_saver)
from src.core.result_saver import ResultSaver
from datetime import datetime
from pathlib import Path

# 创建结果保存器
result_saver = ResultSaver(output_dir="outputs/Palm&Soybean_Results/strategy_runs")

# 1. 保存策略配置
print("  [7.1] 保存策略配置...")

# 提取模型参数
model_params = {
    "model_type": "gradient_boosting",
    "task": model.task,
    "n_estimators": model.model.n_estimators if hasattr(model.model, 'n_estimators') else None,
    "max_depth": model.model.max_depth if hasattr(model.model, 'max_depth') else None,
    "learning_rate": model.model.learning_rate if hasattr(model.model, 'learning_rate') else None,
    "random_state": 42,
    "scaler_used": model.scaler is not None,
    "feature_count": len(selected_features)
}

# 提取信号生成参数
signal_params = {
    "use_rolling_quantile": signal_generator.use_rolling_quantile,
    "rolling_window": signal_generator.rolling_window,
    "upper_quantile": signal_generator.upper_quantile,
    "lower_quantile": signal_generator.lower_quantile,
    "signal_holding_days": signal_generator.signal_holding_days,
    "use_probability": True
}

# 提取回测参数
backtest_params = {
    "initial_capital": backtest_engine.initial_capital,
    "commission_rate": backtest_engine.commission_rate,
    "slippage_rate": backtest_engine.slippage_rate,
    "max_position": backtest_engine.max_position,
    "max_capital_usage": backtest_engine.max_capital_usage
}

# 数据信息
data_info = {
    "start_date": START_DATE,
    "end_date": datetime.now().strftime('%Y-%m-%d'),
    "symbols": list(price_data.keys()),
    "spread_type": "Coking_Profit",
    "train_samples": len(X_train),
    "test_samples": len(X_test),
    "train_period": f"{train_idx.min()} to {train_idx.max()}",
    "test_period": f"{test_idx.min()} to {test_idx.max()}",
    "total_features": len(selected_features),
    "feature_categories": {
        "spread_features": len([f for f in selected_features if 'spread' in f.lower()]),
        "price_features": len([f for f in selected_features if any(s in f for s in ['JM_', 'J_',])]),
        "technical_features": len([f for f in selected_features if any(s in f for s in ['RSI', 'MACD', 'BB'])]),
        "macro_features": len([f for f in selected_features if any(s in f for s in ['VIX', 'DXY', 'RB'])]),
        "fundamental_features": len([f for f in selected_features if any(s in f for s in ['warehouse_receipt', 'inventory', 'inflow'])]),
    }
}

config_file = result_saver.save_strategy_config(
    model_params=model_params,
    signal_params=signal_params,
    backtest_params=backtest_params,
    data_info=data_info
)
print(f"    ✓ 策略配置已保存")

# 2. 保存绩效报告
print("  [7.2] 保存绩效报告...")
perf_files = result_saver.save_performance_report(
    performance_report=performance_report,
    trade_log=trade_log,
    equity_curve=equity_curve
)
print(f"    ✓ 绩效报告已保存:")
for file_type, filepath in perf_files.items():
    print(f"      - {file_type}: {Path(filepath).name}")

# 3. 保存特征信息
print("  [7.3] 保存特征信息...")
feature_file = result_saver.save_feature_info(
    selected_features=selected_features,
    feature_importance=model.feature_importance
)
print(f"    ✓ 特征信息已保存")

# 4. 保存模型指标
print("  [7.4] 保存模型指标...")
train_metrics = model.evaluate(X_train_selected, y_train)
test_metrics = model.evaluate(X_test_selected, y_test)
metrics_file = result_saver.save_model_metrics(
    train_metrics=train_metrics,
    test_metrics=test_metrics
)
print(f"    ✓ 模型指标已保存")
print(f"      训练集准确率: {train_metrics.get('accuracy', 0):.4f}")
print(f"      测试集准确率: {test_metrics.get('accuracy', 0):.4f}")

# 5. 保存交易信号
print("  [7.5] 保存交易信号...")
# 获取概率（如果有）
try:
    if hasattr(model.model, 'predict_proba'):
        probabilities = pd.DataFrame(
            model.model.predict_proba(X_test),
            index=test_idx,
            columns=[f'prob_class_{i}' for i in range(len(model.model.classes_))]
        )
    else:
        probabilities = None
except Exception as e:
    logger.warning(f"无法获取预测概率: {e}")
    probabilities = None

signals_file = result_saver.save_signals(
    signals=signals,
    probabilities=probabilities
)
print(f"    ✓ 交易信号已保存")

# 6. 创建README
print("  [7.6] 创建README...")
result_saver.create_readme()
print(f"    ✓ README已创建")

# 显示输出目录
output_dir = result_saver.get_run_directory()
print(f"\n✓ 所有结果已保存到: {output_dir}")
logger.info(f"所有结果已保存到: {output_dir}\n")

# 打印目录结构
print("\n📂 生成的文件:")
for file in sorted(Path(output_dir).glob('*')):
    size_kb = file.stat().st_size / 1024
    print(f"  📄 {file.name:<35} ({size_kb:>8.2f} KB)")

print("\n" + "="*80)
print("💾 结果保存完成！")
print("="*80)

print("\n📊 快速查看:")
print(f"  绩效汇总: {output_dir}/performance_summary.txt")
print(f"  交易记录: {output_dir}/performance_trades.csv")
print(f"  策略配置: {output_dir}/strategy_config.json")

print("\n💡 提示:")
print("  - 所有CSV文件可以用Excel直接打开")
print("  - JSON文件包含完整的配置和指标信息")
print("  - 不同运行的结果通过时间戳文件夹区分")
print("  - 可以对比不同参数设置的效果")


INFO:__main__:
INFO:__main__:步骤7：保存结果
INFO:__main__:============================================================
INFO:src.core.result_saver:结果保存器初始化完成，输出目录: outputs\Palm&Soybean_Results\strategy_runs\20251103_165242
INFO:src.core.result_saver:策略配置已保存: outputs\Palm&Soybean_Results\strategy_runs\20251103_165242\strategy_config.json
INFO:src.core.result_saver:绩效报告JSON已保存: outputs\Palm&Soybean_Results\strategy_runs\20251103_165242\performance_report.json
INFO:src.core.result_saver:绩效报告CSV已保存: outputs\Palm&Soybean_Results\strategy_runs\20251103_165242\performance_report.csv
INFO:src.core.result_saver:交易日志已保存: outputs\Palm&Soybean_Results\strategy_runs\20251103_165242\performance_trades.csv
INFO:src.core.result_saver:权益曲线已保存: outputs\Palm&Soybean_Results\strategy_runs\20251103_165242\performance_equity_curve.csv
INFO:src.core.result_saver:文本汇总报告已保存: outputs\Palm&Soybean_Results\strategy_runs\20251103_165242\performance_summary.txt
INFO:src.core.result_saver:特征重要性CSV已保存: outputs\Palm&Soybean_


[8/9] 保存策略结果...
  [7.1] 保存策略配置...
    ✓ 策略配置已保存
  [7.2] 保存绩效报告...
    ✓ 绩效报告已保存:
      - report_json: performance_report.json
      - report_csv: performance_report.csv
      - trades: performance_trades.csv
      - equity_curve: performance_equity_curve.csv
      - summary_txt: performance_summary.txt
  [7.3] 保存特征信息...
    ✓ 特征信息已保存
  [7.4] 保存模型指标...
    ✓ 模型指标已保存
      训练集准确率: 0.0000
      测试集准确率: 0.0000
  [7.5] 保存交易信号...
    ✓ 交易信号已保存
  [7.6] 创建README...
    ✓ README已创建

✓ 所有结果已保存到: outputs\Palm&Soybean_Results\strategy_runs\20251103_165242

📂 生成的文件:
  📄 feature_importance.csv              (    1.78 KB)
  📄 feature_info.json                   (    6.85 KB)
  📄 model_metrics.json                  (    0.39 KB)
  📄 performance_equity_curve.csv        (   71.80 KB)
  📄 performance_report.csv              (    0.41 KB)
  📄 performance_report.json             (    0.50 KB)
  📄 performance_summary.txt             (    1.56 KB)
  📄 performance_trades.csv              (   36.54 KB)
  📄 REA

# 超参数优化

使用不同的搜索方法优化模型超参数：
- 网格搜索（Grid Search）：遍历所有参数组合
- 随机搜索（Random Search）：随机采样参数组合
- 贝叶斯优化（Bayesian Optimization）：智能搜索最优参数

## 步骤：
1. 导入超参数优化模块
2. 选择搜索方法
3. 执行参数搜索
4. 比较不同方法的结果
5. 使用最佳参数重新训练模型

In [12]:
# 导入超参数优化模块
from src.core.hyperparameter_tuning import HyperparameterTuner

# 创建超参数优化器
tuner = HyperparameterTuner(
    model_type='gradient_boosting',
    task='classification',
    scoring='f1_weighted',  # 使用加权F1分数
    cv=5,  # 5折交叉验证
    n_jobs=-1,  # 使用所有CPU核心
    verbose=1
)

print("超参数优化器初始化完成")
print(f"模型类型: {tuner.model_type}")
print(f"评分指标: {tuner.scoring}")
print(f"交叉验证折数: {tuner.cv}")

超参数优化器初始化完成
模型类型: gradient_boosting
评分指标: f1_weighted
交叉验证折数: 5


## 方法1：网格搜索（Grid Search）

网格搜索会遍历所有可能的参数组合，找到最优参数。

**优点**：能找到全局最优解（在给定的参数空间内）  
**缺点**：计算成本高，参数组合数呈指数增长  
**适用场景**：参数空间较小，计算资源充足

In [13]:
# # 方法1：网格搜索
# print("\n" + "="*60)
# print("方法1：网格搜索（Grid Search）")
# print("="*60)

# # 创建基础模型
# from sklearn.ensemble import GradientBoostingClassifier
# from src.core.hyperparameter_tuning import HyperparameterTuner
# base_model_grid = GradientBoostingClassifier(random_state=42)

# tuner=HyperparameterTuner(
#     model_type='gradient_boosting',
#     task='classification',
#     cv=5,)
# # 执行网格搜索
# best_params_grid = tuner.grid_search(
#     model=base_model_grid,
#     X_train=X_train,
#     y_train=y_train
# )

# print("\n网格搜索结果:")
# print(f"最佳参数: {best_params_grid}")
# print(f"最佳得分: {tuner.best_score_:.4f}")

# # 显示前10个最佳参数组合
# print("\n前10个最佳参数组合:")
# summary_grid = tuner.get_search_results_summary()
# print(summary_grid.head(10).to_string())

## 使用最佳参数重新训练模型

使用搜索到的最佳参数重新训练模型，并评估性能提升。

In [14]:
# # 使用最佳参数重新训练模型
# print("\n" + "="*60)
# print("使用最佳参数重新训练模型")
# print("="*60)
# best_params_final = best_params_grid
# best_tuner = tuner
# # 选择最佳方法的参数
# # if best_method == 'Grid Search':
# #     best_params_final = best_params_grid
# #     best_tuner = tuner
# # elif best_method == 'Random Search':
# #     best_params_final = best_params_random
# #     best_tuner = tuner_random
# # else:
# #     best_params_final = best_params_bayes
# #     best_tuner = tuner_bayes

# # print(f"\n使用 {best_method} 的最佳参数:")
# for param, value in best_params_final.items():
#     print(f"  {param}: {value}")

# # 使用最佳参数创建新模型
# model_optimized = MLModel(model_type='gradient_boosting', task='classification')

# # 使用最佳参数训练
# print("\n训练优化后的模型...")
# model_optimized.model = GradientBoostingClassifier(**best_params_final, random_state=42)
# model_optimized.model.fit(X_train, y_train)

# # 评估优化后的模型
# metrics_optimized = model_optimized.evaluate(X_test, y_test)

# print("\n优化后模型性能:")
# print(f"  准确率: {metrics_optimized.get('accuracy', 0):.4f}")
# print(f"  F1分数: {metrics_optimized.get('f1', 0):.4f}")
# print(f"  精确率: {metrics_optimized.get('precision', 0):.4f}")
# print(f"  召回率: {metrics_optimized.get('recall', 0):.4f}")

# # 与原始模型比较
# print("\n性能对比:")
# print(f"  原始模型 F1: {metrics.get('f1', 0):.4f}")
# print(f"  优化模型 F1: {metrics_optimized.get('f1', 0):.4f}")
# improvement = (metrics_optimized.get('f1', 0) - metrics.get('f1', 0)) / metrics.get('f1', 1) * 100
# print(f"  提升幅度: {improvement:+.2f}%")

# # 保存优化结果
# best_tuner.save_results('models/tuning_results')
# print("\n✓ 优化结果已保存到 models/tuning_results/")

# # 更新全局model变量为优化后的模型
# model = model_optimized
# print("\n✓ 全局模型已更新为优化后的模型")

In [ ]:
signal_generator = SignalGenerator(model, threshold=0.5, signal_holding_days=20)
signals = signal_generator.generate_signals(X_test, use_probability=True)
signals.index = test_idx
print(f"    ✓ 生成 {len(signals)} 个信号")
print(f"    信号分布: {signals.value_counts().to_dict()}")

# 获取价差价格数据
print("  [5.2] 准备价格数据...")
# 确保spread_data索引与test_idx时区一致
spread_df_for_backtest = spread_data['Coking_Profit'].copy()
if hasattr(spread_df_for_backtest.index, 'tz') and spread_df_for_backtest.index.tz is not None:
    spread_df_for_backtest.index = spread_df_for_backtest.index.tz_localize(None)

spread_prices = spread_df_for_backtest.loc[test_idx, ['spread']].copy()
spread_prices.columns = ['close']
spread_prices['volatility'] = spread_prices['close'].pct_change().rolling(20).std()
print(f"    ✓ 价格数据: {len(spread_prices)} 条")

# 运行回测
print("  [5.3] 运行回测...")
backtest_engine = BacktestEngine(
    initial_capital=1000000,
    commission_rate=0.0005,
    slippage_rate=0.0001,
    max_position=1e16,
    max_capital_usage=0.05
)

equity_curve = backtest_engine.run_backtest(
    spread_prices,
    signals,
    price_col='close',
    volatility_col='volatility'
)
print(f"    ✓ 回测完成，最终权益: ${equity_curve['equity'].iloc[-1]:,.2f}")

# 获取交易日志
trade_log = backtest_engine.get_trade_log()
print(f"    ✓ 总交易次数: {len(trade_log)}")

# 绩效分析
print("  [5.4] 绩效分析...")
analyzer = PerformanceAnalyzer(
    equity_curve,
    initial_capital=1000000,
    risk_free_rate=0.02
)

performance_report = analyzer.generate_performance_report(trade_log)
print(f"    ✓ 总收益率: {performance_report.get('total_return', 0)*100:.2f}%")
print(f"    ✓ 夏普比率: {performance_report.get('sharpe_ratio', 0):.2f}")
print(f"    ✓ 最大回撤: {performance_report.get('max_drawdown', 0)*100:.2f}%")

print("✓ 回测完成")
logger.info("回测完成\n")

# 🔍 调试点6：在此处设置断点，检查回测结果
# 可以查看: equity_curve.tail(), trade_log.head(), performance_report

# ============================================================
# 步骤6：可视化
# ============================================================
print("\n[7/9] 开始可视化...")
logger.info("\n" + "="*60)
logger.info("步骤6：结果可视化")
logger.info("="*60)

print("  [6.1] 生成价格和价差图...")
visualizer.plot_price_and_spread(
    price_data,
    spread_data['Coking_Profit'],
    title='Coking_Profit'
)
print("    ✓ price_spread_chart.png")

print("  [6.2] 生成权益曲线图...")
visualizer.plot_equity_curve(equity_curve)
print("    ✓ equity_curve.png")

print("  [6.3] 生成收益率分布图...")
returns = equity_curve['equity'].pct_change().dropna()
visualizer.plot_returns_distribution(returns)
print("    ✓ returns_distribution.png")

print("  [6.4] 生成月度收益热力图...")
visualizer.plot_monthly_returns_heatmap(equity_curve)
print("    ✓ monthly_returns_heatmap.png")

print("  [6.5] 生成滚动指标图...")
visualizer.plot_rolling_metrics(equity_curve, window=60)
print("    ✓ rolling_metrics.png")

print("  [6.6] 生成交易分析图...")
visualizer.plot_trade_analysis(trade_log)
print("    ✓ trade_analysis.png")

print("✓ 可视化完成")
logger.info("可视化完成\n")

INFO:src.core.ml_models:使用固定阈值模式: 上阈值=0.05, 下阈值=-0.05
INFO:src.core.ml_models:回归信号统计:
INFO:src.core.ml_models:  预测值范围: [-9.5793, 3.0636]
INFO:src.core.ml_models:  做多信号(1): 195 (20.9%)
INFO:src.core.ml_models:  观望信号(0): 26 (2.8%)


INFO:src.core.ml_models:  做空信号(-1): 711 (76.3%)
INFO:src.core.ml_models:生成交易信号完成，信号分布:
INFO:src.core.ml_models:-1    711
 1    195
 0     26
Name: signal, dtype: int64
INFO:src.core.ml_models:应用20天信号维持后，信号分布:
INFO:src.core.ml_models:-1    731
 1    201
Name: signal, dtype: int64


    ✓ 生成 932 个信号
    信号分布: {(-9.579294184083587, -1, 1.0): 1, (-0.3533807406547973, -1, 1.0): 1, (-0.3794884997541652, -1, 1.0): 1, (-0.3792486093420331, -1, 1.0): 1, (-0.37306102171412014, -1, 1.0): 1, (-0.37295935528537677, -1, 1.0): 1, (-0.3659872277123976, -1, 1.0): 1, (-0.3638887545527088, -1, 1.0): 1, (-0.3617917431521986, -1, 1.0): 1, (-0.360657887313367, -1, 1.0): 1, (-0.3595096008623464, -1, 1.0): 1, (-0.3589449011858663, -1, 1.0): 1, (-0.35791861404727077, -1, 1.0): 1, (-0.3578565365048274, -1, 1.0): 1, (-0.34881432632859416, -1, 1.0): 1, (-0.4732404051141878, -1, 1.0): 1, (-0.3476050941622225, -1, 1.0): 1, (-0.34580711459260105, -1, 1.0): 1, (-0.34180902670569624, -1, 1.0): 1, (-0.334779758268477, -1, 1.0): 1, (-0.3330340022615893, -1, 1.0): 1, (-0.33282257783902486, -1, 1.0): 1, (-0.32847576189163796, -1, 1.0): 1, (-0.3270432594067775, -1, 1.0): 1, (-0.3251248135870263, -1, 1.0): 1, (-0.32098833133865545, -1, 1.0): 1, (-0.31729198002146547, -1, 1.0): 1, (-0.3039351806575386

KeyError: "None of [DatetimeIndex(['2022-02-03 05:00:00', '2022-02-04 05:00:00',\n               '2022-02-07 05:00:00', '2022-02-08 05:00:00',\n               '2022-02-09 05:00:00', '2022-02-10 05:00:00',\n               '2022-02-11 05:00:00', '2022-02-14 05:00:00',\n               '2022-02-15 05:00:00', '2022-02-16 05:00:00',\n               ...\n               '2025-10-06 04:00:00', '2025-10-07 04:00:00',\n               '2025-10-08 04:00:00', '2025-10-09 04:00:00',\n               '2025-10-10 04:00:00', '2025-10-13 04:00:00',\n               '2025-10-14 04:00:00', '2025-10-15 04:00:00',\n               '2025-10-16 04:00:00', '2025-10-17 04:00:00'],\n              dtype='datetime64[ns]', name='date', length=932, freq=None)] are in the [index]"